Longitudinal Biomarker Extraction

# Setup

In [ ]:
library(tidyverse)
library(bigrquery)
library(tictoc)
library(data.table)
library(lubridate)

In [ ]:
source("functions.R")

In [ ]:
tic("Time to run script")

# Define and write out the concept_ids for biomarker extractions

In [ ]:
# key for the ancestor concept ids used to extract biomarkers.  
# Note that some descendant concept ids may be filtered out at later steps. 

bm_concepts_key <- c(
    "A1C" = "3004410, 3005673, 4184637",                   
    "Albumin" =  "37036806, 37048896, 37074175, 4017497",                    
    "ALT" =  "37047736, 4146380",           
    "AST" =  "3013721, 3037081, 36305398", 
    "Height" = "3036277, 3023540",
    "Weight" = "3025315, 3013762, 3027492, 3010220",
    "BMI" = "3038553, 4245997",                     
    "BUN" = "37034280, 37057676, 4074649",                           
    "Cl"  = "37025347, 37027122, 37038125, 37045131, 37071140, 37072330, 4019545",                 
    "CRP" = "3010156, 3020460",                                                 
    "HDLC" = "37035033, 4101713",            
    "INR"   = "40779364",                         
    "LDLC"  = "3009966, 3028288, 3028437, 3053341, 3035009",                         
    "Mg"  = "37062081",                           
    "MPV" = "37071701",                           
    "SBP" = "3004249, 3018586, 3035856, 4152194, 40758413",                           
    "Trig" = "3007943, 3022038, 3022192, 3027997", 
    "Albumin_urine" = "3000034, 3012516, 3039775, 4152996", 
    "Creatinine_urine" = "3001349, 3017250, 3037052, 4150621",
    "UACR"  = "3001802, 3034485, 4108431",                         
    "Bicarbonate"   = "37025013, 37052094, 37057819, 37068572, 40483572, 37042332, 37057658, 3009609, 3014094, 3031147, 3010140, 3049133, 3018225, 3037663",          
    "Hematocrit" = "3009542, 3023314",                    
    "TotChol"   = "3027114",                     
    "Serum_Creatinine" = "37029387, 37074896",
    "PA" = "3001110, 3035995, 4230636",  
    "Pain"  = "21494994, 3035819, 4137083, 43055141, 4022240, 4235567, 4234651, 40664844, 3034263, 2617908, 40664654",                      
    "Platelet"  = "3006297, 3007461, 3016682, 3024929, 3031586",                     
    "PO2"  = "3013502, 37026660, 37032414, 37072906, 40762499",                          
    "Pulse" = "3001376, 3003841, 3022318, 3027018, 4301868",  
    "Basophil_count" =  "3006315, 3013429, 3027651",
    "BasoFra" = "3022096, 3013869, 3009797",
    "Eosinophil_count" = "3013115, 3028615, 3009932",
    "EosFra"  = "3006504, 3010457, 3015956",
    "Neutrophil_count" = "3017732, 3013650, 3017501",
    "NeutFra"  = "3018010, 3008342, 3027368",
    "Lymphocyte_count" = "3019198, 3004327, 3003215",
    "LymphFra" = "3002030, 3037511, 3038058",
    "DBP" = "3012888, 3019962, 3034703, 4154790",
    "RBC" = "3020416, 3026361, 3027017",
    "WBC" = "3000905, 3003282, 3010813",
    "Hemoglobin" = "3000963, 3027484",
    "Calcium_bsp" = "37035164, 37057542, 4193434",
    "Potassium" = "37024456, 37039728, 37042511, 37049716, 37055673, 37074594, 4207483",
    "Sodium_bsp" = "37035773, 37036048, 37055139, 37061512, 37074940",
    "MCH" = "40772748",
    "MCHC" = "37045413",
    "MCV" = "37065843",
    "RDW_ratio" = "3002385, 3019897",
    "Prothrombin_time" = "3002417, 3034426",
    "TotalCK" = "3007220",
    "Troponin" = "37073332, 4021291",
    "Glucose" = "3037110, 4144235, 4182052, 4249006, 37035201, 37023397, 37030222, 37034437, 37065054, 37060543", 
    "Uric_Acid" = "3037556, 4313992",
    "Bilirubin_bsp_conjugated" = "37027118, 37028490", 
    "Bilirubin_bsp_total" = "37046070, 37038273",
    "Bilirubin_urine" = "37034025, 37043080" 
)

#get via bm_concepts_key["biomarker"]

Write out the concept id key. 

In [ ]:
concept_ancestor_ids_out <- data.frame("Feature_Name" = names(bm_concepts_key),
                                      "Concept_Ancestor_IDs" = bm_concepts_key)
rownames(concept_ancestor_ids_out) <- NULL
concept_ancestor_ids_out

In [ ]:
write_to_bucket(concept_ancestor_ids_out, "biomarker_concept_ancestors_AOU.csv")

# Pull biomarkers

## Lipids, HbA1c, Glucose

### HDL

In [ ]:
HDL_df <- pull_lab(bm_concepts_key["HDLC"], name = "HDL")
dim(HDL_df)

In [ ]:
HDL_df %>% count(unit_concept_name, sort=T)

HDL_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() +
    xlim(8.5, 120)

HDL_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

HDL_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() +
    xlim(8.5, 120) + 
    theme(legend.position = "bottom") + 
    guides(color = guide_legend(ncol=1))

In [ ]:
HDL <- HDL_df %>%
    filter(!(measurement_concept_id %in% c(3023884, 4019543, 3027939, 3049783, 3023602))) %>%
    filter(unit_concept_name %in% c("mg/dl", "mg/ml", NA)) %>% #mg/ml look ok, probably entered wrong
    filter(value_as_number >= 8.5 & value_as_number <= 120) 
nrow(HDL)

In [ ]:
HDL <- HDL %>% dedup_records() %>% dedup_median()
nrow(HDL)

In [ ]:
fivenum(HDL$value_as_number)
HDL_summ <- HDL %>% group_by(person_id) %>% summarize(n=n())
fivenum(HDL_summ$n)
nrow(HDL_summ)

In [ ]:
write_to_bucket(HDL, "bm_HDL.csv")

In [ ]:
rm(HDL, HDL_df)
gc()

### LDL

In [ ]:
LDL_df <- pull_lab(bm_concepts_key["LDLC"], name = "LDL")
dim(LDL_df)

In [ ]:
LDL_df %>% count(unit_concept_name, sort=T)

LDL_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() +
    xlim(20, 200)

LDL_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

LDL_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() +
    xlim(20, 200) + 
    theme(legend.position ="bottom") + guides(color = guide_legend(ncol=1))

In [ ]:
LDL <- LDL_df %>%
    filter(value_as_number >= 10 & value_as_number <= 500) %>%
    filter(unit_concept_name %in% c("mg/dl", "mg/dl calculated", NA, "mg/ml" )) %>% #mg/ml seem ok
    dedup_records() %>% dedup_median()

nrow(LDL)
fivenum(LDL$value_as_number)
LDL_summ <- LDL %>% group_by(person_id) %>% summarize(n=n())
fivenum(LDL_summ$n)
nrow(LDL_summ)

In [ ]:
write_to_bucket(LDL, "bm_LDL.csv")

In [ ]:
rm(LDL_df, LDL)
gc()

### Total Cholesterol

In [ ]:
totchol_df <- pull_lab(bm_concepts_key["TotChol"], name = "totchol")
dim(totchol_df)

In [ ]:
totchol_df %>% count(unit_concept_name, sort=T)

totchol_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() +
    xlim(50, 600)

totchol_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

totchol_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() +
    xlim(50, 600)

There are an unreasonable number of measures of totchol of exactly 200 and 239...
200 and 240 correspond to the thresholds for 'elevated' and 'high'.   So, they are probably not exact measures.  
To filter for only exactly observed measures, filter out records with NA unit and these exact values (200 and 239).  
It will also unfortunately git rid of some measures which truly were this exact level. 

In [ ]:
totchol <- totchol_df %>%
    filter(!(value_as_number %in% c(200, 239)  &  is.na(unit_concept_name)))

In [ ]:
totchol <- totchol %>%
    filter(value_as_number >= 50 & value_as_number <= 600) %>%
    filter(unit_concept_name %in% c("mg/dl", "g/dl", "mg/ml", NA)) %>% # "g/dl" and "mg/ml" seem ok
    dedup_records() %>% dedup_median()

nrow(totchol)
fivenum(totchol$value_as_number)
totchol_summ <- totchol %>% group_by(person_id) %>% summarize(n=n())
fivenum(totchol_summ$n)
nrow(totchol_summ)

In [ ]:
write_to_bucket(totchol, "bm_total_cholesterol.csv")

In [ ]:
rm(totchol_df, totchol)
gc()

### Triglycerides

In [ ]:
triglycerides_df <- pull_lab(bm_concepts_key["Trig"], name = "triglycerides")
dim(triglycerides_df)

In [ ]:
triglycerides_df %>% count(unit_concept_name, sort=T)

triglycerides_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() +
    xlim(20, 600)

triglycerides_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

triglycerides_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() +
    xlim(20, 600)

In [ ]:
triglycerides <- triglycerides_df %>%
    filter(value_as_number >= 15 & value_as_number <= 2000) %>%
    filter(unit_concept_name %in% c("mg/dl", "mg/ml", NA )) %>% # mg/ml seems ok
    dedup_records() %>% dedup_median()

nrow(triglycerides)
fivenum(triglycerides$value_as_number)
triglycerides_summ <- triglycerides %>% group_by(person_id) %>% summarize(n=n())
fivenum(triglycerides_summ$n)
nrow(triglycerides_summ)

In [ ]:
write_to_bucket(triglycerides, "bm_triglycerides.csv")

In [ ]:
rm(triglycerides, triglycerides_df)
gc()

### HbA1c

In [ ]:
HbA1c_df <- pull_lab(bm_concepts_key["A1C"], name = "HbA1c")
dim(HbA1c_df)

In [ ]:
HbA1c_df %>% count(unit_concept_name, sort=T)

HbA1c_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() +
    xlim(4, 12)

HbA1c_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

HbA1c_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() +
    xlim(4, 12)

In [ ]:
HbA1c <- HbA1c_df %>%
    filter(value_as_number >= 3 & value_as_number <= 20) %>%
    filter(unit_concept_name %in% c("percent", NA,"of total h", "percentage unit", "percent hemoglobin",
                                    "percent hemoglobin a1c", "% of total", "percentage of total",
                                   "percentage total hemoglobin"))  %>%
    dedup_records() %>% dedup_median() %>% 
    filter(summarized_from_n <= 2)
nrow(HbA1c)
fivenum(HbA1c$value_as_number)
HbA1c_summ <- HbA1c %>% group_by(person_id) %>% summarize(n=n())
fivenum(HbA1c_summ$n)
nrow(HbA1c_summ)

In [ ]:
write_to_bucket(HbA1c, "bm_HbA1c.csv")

In [ ]:
rm(HbA1c_df, HbA1c)
gc()

### Glucose (fasting vs. random)

In [ ]:
bm_concepts_key["Glucose"]
Glucose_df <- pull_lab(bm_concepts_key["Glucose"], name = "Glucose")
dim(Glucose_df)

In [ ]:
Glucose_df %>% count(unit_concept_name, sort=T)

In [ ]:
Glucose <- Glucose_df %>% filter(unit_concept_name %in% c("mg/dl", NA, "mg/ml" )) #exclude moles

In [ ]:
# categorize topography and fasting status, whether the concept name says it is measured in 
# moles(although unit says otherwise)

glucose_key <- Glucose %>%
    count(measurement_concept_id, standard_concept_name) %>% 
    arrange(desc(n)) %>% 
    mutate(maybe_moles = grepl("mole|Mole", standard_concept_name),
          glucose_type = case_when(
              grepl("Capillary|capillary|strip", standard_concept_name) ~ "FingerStick",
              grepl("Serum, Plasma or Blood", standard_concept_name) ~ "Unknown",
              grepl("Serum or Plasma|serum|plasma", standard_concept_name, ignore.case=T) ~ "Serum_Plasma",
              grepl("Blood|blood", standard_concept_name) ~ "Blood",
              TRUE ~ "Unknown"),
          glucose_fasting = grepl("Fast|fast", standard_concept_name)) 

glucose_key

In [ ]:
Glucose <- Glucose %>% left_join(glucose_key) 

In [ ]:
Glucose %>% 
    ggplot(aes(x=value_as_number, color = glucose_fasting, linetype = maybe_moles)) + 
    geom_density() + xlim(30, 300) +
    facet_wrap(unit_concept_name~glucose_type, scales="free_y") +
    theme(legend.position = "bottom") +
    guides(color = guide_legend(ncol=2, byrow=T))

In [ ]:
Glucose %>% filter(glucose_type == "Serum_Plasma") %>%
    group_by(standard_concept_name) %>%
    filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(30, 300) + 
    theme(legend.position="bottom") + guides(color = guide_legend(ncol=1))

In [ ]:
Glucose %>% filter(glucose_type == "FingerStick") %>%
    group_by(standard_concept_name) %>%
    filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(30, 300) + 
    theme(legend.position="bottom") + guides(color = guide_legend(ncol=1))

In [ ]:
Glucose %>% filter(glucose_type == "Blood") %>%
    group_by(standard_concept_name) %>%
    filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(30, 300) + 
    theme(legend.position="bottom") + guides(color = guide_legend(ncol=1))

In [ ]:
Glucose %>% filter(glucose_type == "Unknown") %>%
    group_by(standard_concept_name) %>%
    filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(30, 300) + 
    theme(legend.position="bottom") + guides(color = guide_legend(ncol=1))

In [ ]:
Glucose <- Glucose %>% filter(!(measurement_concept_id %in% c(3014305, 4120298)))

Those supposedly measured in moles are consisent with really being in grams.  Those with moles as unit were already removed. 

In [ ]:
prop.table(table(Glucose$glucose_type)) %>% signif(3)

In [ ]:
tic()
Glucose <- Glucose %>%
    filter(value_as_number >= 30 & value_as_number <= 1500) %>%
    dedup_records() %>% dedup_median()
nrow(Glucose_df)
nrow(Glucose)
fivenum(Glucose$value_as_number)
Glucose_summ <- Glucose %>% group_by(person_id) %>% summarize(n=n())
fivenum(Glucose_summ$n)
nrow(Glucose_summ)
toc()

In [ ]:
Glucose <- Glucose %>% left_join(glucose_key)

In [ ]:
write_to_bucket(Glucose, "bm_Glucose.csv")

In [ ]:
rm(Glucose_df, Glucose)
gc()

## Blood Pressure, BMI, Vitals, Pain

### SBP

In [ ]:
SBP_df <- pull_lab(bm_concepts_key["SBP"], name = "SBP")
dim(SBP_df)
gc()

In [ ]:
SBP_df %>% count(unit_concept_name, sort=T)

SBP_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() +
    xlim(50, 300)

SBP_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

SBP_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() +
    xlim(50, 300)

In [ ]:
#Note that some Patients got technical duplicates / triplicates of the measures at their visit.  
SBP <- SBP_df %>%
    filter(value_as_number >= 50 & value_as_number <= 300) %>% 
    mutate(unit_concept_name = "mmHg") %>% #easier dedup
    dedup_records() %>% dedup_median()

nrow(SBP)
fivenum(SBP$value_as_number)
SBP_summ <- SBP %>% group_by(person_id) %>% summarize(n=n())
fivenum(SBP_summ$n)
nrow(SBP_summ)

In [ ]:
write_to_bucket(SBP, "bm_SBP.csv") 

In [ ]:
rm(SBP_df, SBP_summ) #keep SBP in memory to join with DBP
gc()

### DBP

In [ ]:
DBP_df <- pull_lab(bm_concepts_key["DBP"], name = "DBP", read_existing = TRUE)
dim(DBP_df)

In [ ]:
DBP_df %>% count(unit_concept_name, sort=T) 

DBP_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() +
    xlim(30, 200)

DBP_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

DBP_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() +
    xlim(30, 200)

In [ ]:
DBP <- DBP_df %>%
    filter(value_as_number >= 30 & value_as_number <= 200) %>% 
    mutate(unit_concept_name = "mmHg") %>% #easier dedup
    dedup_records() %>% dedup_median()

nrow(DBP)
fivenum(DBP$value_as_number)
DBP_summ <- DBP %>% group_by(person_id) %>% summarize(n=n())
fivenum(DBP_summ$n)
nrow(DBP_summ)

In [ ]:
rm(DBP_df)
gc()

In [ ]:
write_to_bucket(DBP, "bm_DBP.csv")

### Joined SBP and DBP

In [ ]:
#Join SBP and DBP by person and datetime
tic()
BP <- SBP %>%
    distinct(person_id, measurement_date, SBP = value_as_number, visit_occurrence_concept_name) %>%
    inner_join(DBP %>% 
                distinct(person_id, measurement_date, DBP = value_as_number, 
                       visit_occurrence_concept_name2 = visit_occurrence_concept_name)) %>%
    mutate(visit_occurrence_concept_name = coalesce(visit_occurrence_concept_name, visit_occurrence_concept_name2)) %>%
    select(-visit_occurrence_concept_name2) %>%
    arrange(person_id, measurement_date) %>%
    distinct(person_id, measurement_date, SBP, DBP, visit_occurrence_concept_name)
toc()
nrow(BP)
length(unique(BP$person_id)) 

In [ ]:
write_to_bucket(BP, "bm_BP.csv")

In [ ]:
rm(SBP, DBP, BP)
gc()

### Height, Weight, and BMI

#### Height

In [ ]:
Height_df <- pull_lab(bm_concepts_key["Height"], name = "height")
dim(Height_df)

In [ ]:
Height_df %>% count(unit_concept_name, sort=T)

Height_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(100, 250)

Height_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

Height_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density()  + xlim(100, 250)

In [ ]:
Height <- Height_df %>%
    filter(unit_concept_name == "cm") %>%
    filter(value_as_number >= 100 & value_as_number <= 250) %>% 
    distinct(person_id, measurement_date, value_as_number) %>%
    group_by(person_id, measurement_date) %>%
    summarize(value_as_number = median(value_as_number)) %>%
    ungroup()

nrow(Height)
fivenum(Height$value_as_number)
Height_summ <- Height  %>% group_by(person_id) %>% summarize(n=n())
fivenum(Height_summ$n)
nrow(Height_summ)

In [ ]:
Height_Median = Height %>%
  group_by(person_id) %>%
  summarise(median_height = median(value_as_number/100, na.rm=T)) %>%
    ungroup()

fivenum(Height_Median$median_height)
nrow(Height_Median)

#### Weight

In [ ]:
Weight_df <- pull_lab(bm_concepts_key["Weight"], name = "weight")
dim(Weight_df)

In [ ]:
Weight_df %>% count(unit_concept_name, sort=T)
      
Weight_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(40, 250)

Weight_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

Weight_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(40, 250)

In [ ]:
Weight <- Weight_df %>% 
    mutate(value_as_number = ifelse( (!is.na(unit_concept_name) & unit_concept_name == "pound (us)") | 
                                    measurement_concept_id == 3010220, 
                                    value_as_number*0.453592, value_as_number)) %>%
    filter(unit_concept_name %in% c("kilog", "pound (us)"))

In [ ]:
Weight %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(40, 175)

Weight %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(40, 175)

In [ ]:
Weight <- Weight %>%
    #mutate(unit_concept_name = "kg") %>%
    distinct(person_id, measurement_date, value_as_number) %>%
    group_by(person_id, measurement_date) %>%
    summarize(value_as_number = median(value_as_number)) %>%
    ungroup()

nrow(Weight)
fivenum(Weight$value_as_number)
Weight_summ <- Weight %>% group_by(person_id) %>% summarize(n=n())
fivenum(Weight_summ$n)
nrow(Weight_summ)

In [ ]:
computed_BMI= Weight %>%
  left_join(Height_Median) %>%
  mutate(BMI = round(value_as_number / median_height / median_height, 1)) %>%
  distinct(person_id, measurement_date, BMI) %>%
  filter(BMI <= 100) %>%
  filter(BMI > 10) %>%
  arrange(person_id, measurement_date)

nrow(computed_BMI)
length(unique(computed_BMI$person_id))
fivenum(computed_BMI$BMI)

#### BMI direct from EHR

In [ ]:
BMIraw_df <- pull_lab(bm_concepts_key["BMI"], name = "BMIraw")
dim(BMIraw_df)

In [ ]:
BMIraw_df %>% count(unit_concept_name, sort=T)

BMIraw_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(10, 100)

BMIraw_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

BMIraw_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(10, 100)

In [ ]:
BMIraw <- BMIraw_df %>%
    filter(value_as_number >= 10 & value_as_number <= 100) %>%
    mutate(value_as_number = round(value_as_number, 1)) %>%
    distinct(person_id, measurement_date, value_as_number) %>%
    group_by(person_id, measurement_date) %>%
    summarize(value_as_number = median(value_as_number)) %>%
    ungroup()  %>%
    arrange(person_id, measurement_date) %>%
    dplyr::rename(BMI = value_as_number) %>% 
    mutate(BMI = round(BMI, 1))

nrow(BMIraw)
fivenum(BMIraw$BMI)
BMIraw_summ <- BMIraw %>% group_by(person_id) %>% summarize(n=n())
fivenum(BMIraw_summ$n)
nrow(BMIraw_summ)

#### Combine direct and calculated BMI

In [ ]:
BMI_compare <- computed_BMI %>%
    full_join(BMIraw %>% dplyr::rename(BMIdirect = BMI))

In [ ]:
BMI_compare %>% group_by(is.na(BMI), is.na(BMIdirect)) %>% count()
# we get 3.6M extra BMI measures by also calculating ourselves

In [ ]:
BMI_compare <- BMI_compare %>% mutate(diff = BMIdirect - BMI)
hist(BMI_compare$diff, nclass = 1000, xlim = c(-10, 10))
mean(BMI_compare$diff, na.rm=T)
fivenum(BMI_compare$diff)

In [ ]:
BMI_compare %>% filter(abs(diff) > 2) %>% select(BMI, BMIdirect) %>%
    pivot_longer(everything(), names_to = "BMI_type") %>%
    ggplot(aes(x=value, color = BMI_type)) + geom_density()

In [ ]:
BMI_compare %>% filter(abs(diff) > 2) %>% 
    ggplot(aes(x=BMI, y=BMIdirect)) + geom_point(size=0.1) + 
    geom_abline(aes(intercept = 0, slope = 1/.453), color = 'red') #pounds wrongly recorded as kg?

In [ ]:
chck <- BMI_compare %>% filter(abs(diff) > 3) %>% distinct(person_id) %>%
    sample_n(16) %>% left_join(BMI_compare) %>% mutate(badpt = abs(diff) > 3)

In [ ]:
ggplot(chck) + geom_line(aes(x=measurement_date, y = BMI), color = "blue") + #maybe more reliable?
    geom_line(aes(x=measurement_date, y = BMIdirect), color = "green") + 
    facet_wrap(~person_id, scales="free")

In [ ]:
BMI_final <- BMI_compare %>%
    filter(is.na(diff) | abs(diff) <=2 ) %>% # only keep the record if they agree
    mutate(BMI = coalesce(BMIdirect, BMI)) %>% # then prefer the one from EHR rather than calculated
    select(-BMIdirect, -diff) %>%
    distinct()
nrow(BMI_final)

In [ ]:
write_to_bucket(BMI_final, "bm_BMI.csv")

In [ ]:
suppressWarnings(
    rm(BMI_compare, BMIraw_df, BMIraw, computed_BMI, Height, Height_df, Height_Median, Weight, Weight_df, BMI_final))
gc();

### Pain

In [ ]:
bm_concepts_key["Pain"]
#note this includes concepts to be extracted from both measurements and observations domains

In [ ]:
Pain_df <- pull_lab(bm_concepts_key["Pain"], name = "Pain", remove_NA = FALSE) # we may be able to impute some with 0
dim(Pain_df)

In [ ]:
Pain_df %>% count(value_as_number) %>% arrange(desc(n)) %>% head(20)

In [ ]:
Pain_df_observation <- pull_observation_with_value(bm_concepts_key["Pain"], name = "Pain") #NAs not removed by default
dim(Pain_df_observation)

In [ ]:
mean(is.na(Pain_df_observation$value_as_number))
Pain_df_observation %>% count(value_as_number, sort=T) %>% head(20)

Clean data from measurement domain

In [ ]:
Pain_df %>% count(unit_concept_name, sort=T)

Pain_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(0,10)

Pain_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

Pain_df %>% filter(value_as_number > 0) %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density()  + xlim(0,10)

In [ ]:
mean(is.na(Pain_df$value_as_number))

Clean data from observation domain

In [ ]:
Pain_df_observation %>% count(unit_concept_name, sort=T)

Pain_df_observation %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density(bw=.1) + xlim(0,10)

Pain_df_observation %>% group_by(observation_concept_id, standard_concept_name) %>% count() %>% arrange(desc(n))

Pain_df_observation %>% filter(value_as_number > 0) %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density(bw=.1)  + xlim(0,10)

In [ ]:
Pain_df_observation %>% filter(is.na(Pain_df_observation$value_as_number)) %>% 
    count(observation_concept_id, standard_concept_name) %>% arrange(desc(n))

In [ ]:
Pain_df_observation %>% filter(is.na(Pain_df_observation$value_as_number)) %>% 
    count(value_as_string, sort=T)

In [ ]:
Pain_observation <- Pain_df_observation %>%
    mutate(value_as_number = case_when(!is.na(value_as_number) ~ value_as_number,
                                       observation_concept_id %in% c(40664844) ~ 0,
                                      TRUE ~ NA)) %>%
    select(-value_as_string, -qualifier_concept_name, -observation_type_concept_name) %>%
    filter(!is.na(value_as_number)) %>%
    dplyr::rename(measurement_concept_id = observation_concept_id)
dim(Pain_observation)

In [ ]:
Pain_combined <- full_join(Pain_df, Pain_observation) %>%
    filter(value_as_number %in% 0:10) %>% #only keep integers
    mutate(unit_concept_name = "score")
dim(Pain_combined)

In [ ]:
 Pain <- Pain_combined %>%
    dedup_records() %>% dedup_median()

nrow(Pain)
fivenum(Pain$value_as_number)
Pain_summ <- Pain %>% group_by(person_id) %>% summarize(n=n())
fivenum(Pain_summ$n)
nrow(Pain_summ)

In [ ]:
write_to_bucket(Pain, "bm_Pain.csv")

In [ ]:
rm(Pain, Pain_df)
gc()

### PO2 / SaO2

In [ ]:
# this variably maybe not aptly named in KDI
bm_concepts_key["PO2"]

In [ ]:
SaO2_df <- pull_lab(bm_concepts_key["PO2"], name = "SaO2")
dim(SaO2_df)

In [ ]:
SaO2_df %>% count(unit_concept_name, sort=T)

SaO2_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density(bw=1) + xlim(50, 100)

SaO2_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

SaO2_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density(bw=1)  + xlim(50, 100) + 
    theme(legend.position="bottom") + 
    guides(color=guide_legend(ncol=1))

In [ ]:
SaO2 <- SaO2_df %>%
    filter(!(measurement_concept_id %in% c(42869600, 3041253))) %>% #exclude venous blood
    filter(unit_concept_name %in% c("percent", "percent hemoglobin", "percent saturation", 
                                    "volume percent", "percentage unit", NA)) %>%
    filter(value_as_number >= 80 & value_as_number <= 100) %>%
    mutate(unit_concept_name = "percent") %>% #easier dedup
    dedup_records()  %>% dedup_median()

nrow(SaO2)
fivenum(SaO2$value_as_number)
SaO2_summ <- SaO2 %>% group_by(person_id) %>% summarize(n=n())
fivenum(SaO2_summ$n)
nrow(SaO2_summ)

In [ ]:
write_to_bucket(SaO2, "bm_SaO2.csv")

In [ ]:
rm(SaO2, SaO2_df)
gc()

### Pulse / Heart Rate

Note that this one takes a while 

In [ ]:
bm_concepts_key["Pulse"]

In [ ]:
HeartRate_df <- pull_lab(bm_concepts_key["Pulse"], name = "HeartRate")
dim(HeartRate_df)

In [ ]:
HeartRate_df %>% count(unit_concept_name, sort=T)

HeartRate_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(15, 250)

HeartRate_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

HeartRate_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density()  + xlim(15, 250)

In [ ]:
gc()

In [ ]:
HeartRate <- HeartRate_df %>%
    filter(value_as_number >= 25 & value_as_number <= 175) %>%
    mutate(unit_concept_name = "beats/min") %>% #easier dedup
    dedup_records() %>% dedup_median()

In [ ]:
nrow(HeartRate)
fivenum(HeartRate$value_as_number)
HeartRate_summ <- HeartRate %>% group_by(person_id) %>% summarize(n=n())
fivenum(HeartRate_summ$n)
nrow(HeartRate_summ)

In [ ]:
write_to_bucket(HeartRate, "bm_HeartRate.csv")

In [ ]:
rm(HeartRate, HeartRate_df)
gc()

## CBC etc, differential

### Red Blood Cells (Erythrocytes)

In [ ]:
RBC_df <- pull_lab(bm_concepts_key["RBC"], name = "RBC")
dim(RBC_df)

In [ ]:
RBC_df %>% count(unit_concept_name, sort=T)

RBC_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() +
    xlim(2, 8)

RBC_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

RBC_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() +
    xlim(2, 8)

In [ ]:
RBC <- RBC_df %>%
    filter(value_as_number >= 2 & value_as_number <= 8) %>%
    filter(unit_concept_name %in% c( "million/microl", NA, "m/microl", "thousand/cubic mm", "m/cubic mm", "per microl", 
                                    "microunit/l",
                                   "nl", "per cubic mm", "microlitre/ml", "million"
                                   )) %>% 
    mutate(unit_concept_name = ifelse(unit_concept_name %in% c("million/microl", "m/microl", "m/cubic mm", "per microl",
                                                              "per cubic mm", "microlitre/ml", "million"), 
                                      "million/microL", unit_concept_name)) %>%
    dedup_records() %>% dedup_median()

nrow(RBC)
fivenum(RBC$value_as_number)
RBC_summ <- RBC %>% group_by(person_id) %>% summarize(n=n())
fivenum(RBC_summ$n)
nrow(RBC_summ)

In [ ]:
write_to_bucket(RBC, "bm_RBC.csv")

In [ ]:
rm(RBC_df, RBC)
gc()

### White Blood Cells (Leukocytes)

In [ ]:
WBC_df <- pull_lab(bm_concepts_key["WBC"], name = "WBC")
dim(WBC_df)

In [ ]:
WBC_df %>% count(unit_concept_name, sort=T)

WBC_df %>% group_by(unit_concept_name) %>% filter(n() > 500) %>%
    ggplot(aes(x=log(value_as_number, 10), color = unit_concept_name)) + 
    geom_density()

In [ ]:
WBC_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

WBC_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=log(value_as_number, 10), color = standard_concept_name)) + 
    geom_density() +
    xlim(0, 2)

In [ ]:
tic()
WBC <- WBC_df %>%
    mutate(value_as_number = ifelse(!is.na(unit_concept_name) & 
                                    unit_concept_name == "per cubic mm" & value_as_number > 100, 
                                    value_as_number/1000, value_as_number)) %>%
    mutate(unit_concept_name = ifelse(unit_concept_name %in% c("per cubic mm", "kelvin/cubic mm",
                                                              "thousand/cubic mm", "thousand", 
                                                               "ul", "microl",
                                                              "/mm3", "cubic mm"), 
                                      "thousand/microl", unit_concept_name)) %>%
    filter(value_as_number >= 1 & value_as_number <= 100) %>%
    filter(unit_concept_name %in% c("thousand/microl", "thousand/ml", "nl", NA)) %>%  
    dedup_records() %>% dedup_median()

nrow(WBC)
fivenum(WBC$value_as_number)
WBC_summ <- WBC %>% group_by(person_id) %>% summarize(n=n())
fivenum(WBC_summ$n)
nrow(WBC_summ)
toc()

"thousand per milliliter" is likely incorrect. 
"per cubic millimeter" needed to be converted. 

In [ ]:
write_to_bucket(WBC, "bm_WBC.csv")

In [ ]:
rm(WBC_df, WBC)
gc()

### Platelets (Platelet Count)

In [ ]:
platelets_df <- pull_lab(bm_concepts_key["Platelet"], name = "platelets")                   
dim(platelets_df)

In [ ]:
platelets_df %>% count(unit_concept_name, sort=T)

platelets_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=log(value_as_number), color = unit_concept_name)) + 
    geom_density()  +
    xlim(log(10), log(4000))

platelets_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

platelets_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=log(value_as_number), color = standard_concept_name)) + 
    geom_density() +
    xlim(log(10), log(4000))

"thousand/ml" and "nl" likely recorded wrong.  Distribution matches that of K/uL.  

In [ ]:
platelets <- platelets_df %>%
    filter(value_as_number >= 10 & value_as_number <= 4000) %>%
    filter(unit_concept_name %in% c("thousand/microl", NA, "nl", 
                                   "x10(3)/mcl", "microl", "per cubic mm",
                                    "kelvin/microl", "thousand", "thousand/cubic mm",
                                    "/mm3", "thousand/ml")) %>%
    mutate(unit_concept_name = ifelse(!is.na(unit_concept_name), "thousand/uL", NA)) %>%
    dedup_records() %>% dedup_median() 

nrow(platelets)
fivenum(platelets$value_as_number)
platelets_summ <- platelets %>% group_by(person_id) %>% summarize(n=n())
fivenum(platelets_summ$n)
nrow(platelets_summ)

In [ ]:
write_to_bucket(platelets, "bm_platelets.csv")

In [ ]:
rm(platelets_df, platelets)
gc()

### Mean Platelet Volume (MPV)

In [ ]:
MPV_df <- pull_lab(bm_concepts_key["MPV"], name = "MPV")
dim(MPV_df)

In [ ]:
MPV_df %>% count(unit_concept_name, sort=T)

MPV_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(5, 15)

MPV_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

MPV_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(5, 15)

In [ ]:
MPV <- MPV_df %>%
    filter(unit_concept_name %in% c("femtol", "fl", NA)) %>%
    mutate(unit_concept_name = "femtoL") %>%
    filter(value_as_number >= 5 & value_as_number <= 15) %>%
    dedup_records() %>% dedup_median()

nrow(MPV)
fivenum(MPV$value_as_number)
MPV_summ <- MPV %>% group_by(person_id) %>% summarize(n=n())
fivenum(MPV_summ$n)
nrow(MPV_summ)

In [ ]:
write_to_bucket(MPV, "bm_MPV.csv")

In [ ]:
rm(MPV, MPV_df)
gc()

### Hematocrit

In [ ]:
hematocrit_df <- pull_lab(bm_concepts_key["Hematocrit"], name = "hematocrit")                            
dim(hematocrit_df)

In [ ]:
hematocrit_df %>% count(unit_concept_name, sort=T)

hematocrit_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() +
    xlim(15, 75)

hematocrit_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

hematocrit_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() +
    xlim(15, 75)

In [ ]:
hematocrit <- hematocrit_df %>%
    filter(value_as_number >= 15 & value_as_number <= 75) %>%
    filter(unit_concept_name %in% c("percent", NA )) %>%
    dedup_records() %>% dedup_median() 

nrow(hematocrit)
fivenum(hematocrit$value_as_number)
hematocrit_summ <- hematocrit %>% group_by(person_id) %>% summarize(n=n())
fivenum(hematocrit_summ$n)
nrow(hematocrit_summ)

In [ ]:
write_to_bucket(hematocrit, "bm_hematocrit.csv")

In [ ]:
rm(hematocrit_df, hematocrit)
gc()

### Hemoglobin

In [ ]:
hemoglobin_df <- pull_lab(bm_concepts_key["Hemoglobin"], name = "hemoglobin")
dim(hemoglobin_df)

In [ ]:
hemoglobin_df %>% count(unit_concept_name, sort=T)

hemoglobin_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() +
    xlim(5, 250)

hemoglobin_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

hemoglobin_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() +
    xlim(5, 25)

In [ ]:
hemoglobin <- hemoglobin_df %>%
    mutate(value_as_number = ifelse(!is.na(unit_concept_name) & unit_concept_name == "g/l", 
                                    value_as_number/10, value_as_number),
          unit_concept_name = ifelse(unit_concept_name == "g/l", "g/dl", unit_concept_name)) %>%
    filter(value_as_number >= 5 & value_as_number <= 25) %>%
    filter(unit_concept_name %in% c("g/dl", NA )) %>%
    dedup_records() %>% dedup_median()

nrow(hemoglobin)
fivenum(hemoglobin$value_as_number)
hemoglobin_summ <- hemoglobin %>% group_by(person_id) %>% summarize(n=n())
fivenum(hemoglobin_summ$n)
nrow(hemoglobin_summ)

In [ ]:
write_to_bucket(hemoglobin, "bm_hemoglobin.csv")

In [ ]:
rm(hemoglobin_df, hemoglobin)
gc()

### MCH

In [ ]:
MCH_df <- pull_lab(bm_concepts_key["MCH"], name = "MCH")    
dim(MCH_df)

In [ ]:
MCH_df %>% count(unit_concept_name, sort=T)

MCH_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(10,50)

MCH_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

MCH_df %>% group_by(standard_concept_name) %>% 
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(10,50)

In [ ]:
MCH <- MCH_df %>%
    filter(unit_concept_name %in% c("picog", NA, "picog/cell", "microg")) %>%
    filter(!(measurement_concept_id %in% c(3051341))) %>%
    filter(value_as_number >= 10 & value_as_number <= 50) %>% 
    dedup_records() %>% dedup_median()
nrow(MCH)
fivenum(MCH$value_as_number)
MCH_summ <- MCH %>% group_by(person_id) %>% summarize(n=n())
fivenum(MCH_summ$n)
nrow(MCH_summ)

In [ ]:
write_to_bucket(MCH, "bm_MCH.csv")

In [ ]:
rm(MCH, MCH_df)
gc()

### MCHC

In [ ]:
MCHC_df <- pull_lab(bm_concepts_key["MCHC"], name = "MCHC") 
dim(MCHC_df)

In [ ]:
MCHC_df %>% count(unit_concept_name, sort=T)

MCHC_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(25, 45)

MCHC_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

MCHC_df %>% group_by(standard_concept_name) %>% 
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(25, 45)

In [ ]:
MCHC <- MCHC_df %>%
    filter(unit_concept_name %in% c("g/dl", "g/dl calculated", NA, "percentage unit")) %>%
    mutate(unit_concept_name = "g/dl") %>%
    filter(value_as_number >= 25 & value_as_number <= 45) %>% 
    dedup_records() %>% dedup_median()
nrow(MCHC)
fivenum(MCHC$value_as_number)
MCHC_summ <- MCHC %>% group_by(person_id) %>% summarize(n=n())
fivenum(MCHC_summ$n)
nrow(MCHC_summ)

In [ ]:
write_to_bucket(MCHC, "bm_MCHC.csv")

In [ ]:
rm(MCHC, MCHC_df)
gc()

### MCV

In [ ]:
MCV_df <- pull_lab(bm_concepts_key["MCV"], name = "MCV")
dim(MCV_df)

In [ ]:
MCV_df %>% count(unit_concept_name, sort=T)

MCV_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(40, 200)

MCV_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

MCV_df %>% group_by(standard_concept_name) %>% 
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(40, 200)

In [ ]:
MCV <- MCV_df %>%
    filter(unit_concept_name %in% c("femtol", NA)) %>%
    filter(value_as_number >= 40 & value_as_number <= 200) %>% 
    dedup_records() %>% dedup_median()
nrow(MCV)
fivenum(MCV$value_as_number)
MCV_summ <- MCV %>% group_by(person_id) %>% summarize(n=n())
fivenum(MCV_summ$n)
nrow(MCV_summ)

In [ ]:
write_to_bucket(MCV, "bm_MCV.csv")

In [ ]:
rm(MCV, MCV_df)
gc()

### RDW Ratio

In [ ]:
RDW_ratio_df <- pull_lab(bm_concepts_key["RDW_ratio"], name = "RDW_ratio")
dim(RDW_ratio_df)

In [ ]:
RDW_ratio_df %>% count(unit_concept_name, sort=T)

RDW_ratio_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() +xlim(0, 50)

RDW_ratio_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

RDW_ratio_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() +xlim(0, 50)

In [ ]:
RDW_ratio <- RDW_ratio_df %>%
    filter(unit_concept_name %in% c("percent", NA, "unit")) %>%
    filter(value_as_number >= 4 & value_as_number <= 40) %>% 
    dedup_records()  %>% dedup_median()

nrow(RDW_ratio)
fivenum(RDW_ratio$value_as_number)
RDW_ratio_summ <- RDW_ratio %>% group_by(person_id) %>% summarize(n=n())
fivenum(RDW_ratio_summ$n)
nrow(RDW_ratio_summ)

In [ ]:
write_to_bucket(RDW_ratio, "bm_RDW_ratio.csv")

In [ ]:
rm(RDW_ratio, RDW_ratio_df)
gc()

### Basophil count

In [ ]:
bm_concepts_key["Basophil_count"]

In [ ]:
Basophil_count_df <- pull_lab(bm_concepts_key["Basophil_count"], name = "Basophil_ct")                          
dim(Basophil_count_df)

In [ ]:
Basophil_count_df %>% count(unit_concept_name, sort=T)

Basophil_count_df %>% group_by(unit_concept_name) %>% filter(n() > 500) %>%
    ggplot(aes(x=log10(value_as_number), color = unit_concept_name)) + 
    geom_density(bw=.1) 

In [ ]:
Basophil_count <- Basophil_count_df %>%
    mutate(value_as_number = ifelse(is.na(unit_concept_name), value_as_number,
                                         ifelse(unit_concept_name %in% c("cells/microl", "cells/ul", "per microl", "ul"), 
                                                value_as_number/1000, value_as_number))) %>%
    mutate(unit_concept_name = ifelse(unit_concept_name %in% 
                                            c("thousand/cubic mm", "kelvin/microl",  "kelvin/cubic mm", #"billion/l",
                                             "cells/microl", "cells/ul", "per microl", "thousand/microl", "thousand", 
                                              "cubic mm", "kelvin/cubic mm", "ul"), 
                                      "thousand/ul", 
                                      unit_concept_name)) 

Basophil_maybe_frac <- Basophil_count_df %>% filter(unit_concept_name %in% 
                                                    c("percent", "percentage unit", "percent of white blood cells"))

Basophil_count <- Basophil_count %>% anti_join(Basophil_maybe_frac)

In [ ]:
Basophil_count %>% count(unit_concept_name, sort=T) 

Basophil_count %>% group_by(unit_concept_name) %>% filter(n() > 250) %>%
    ggplot(aes(x=log10(value_as_number), color = unit_concept_name)) + 
    geom_density(bw=.1) + xlim(-2.5, -.301)

In [ ]:
Basophil_count <- Basophil_count %>% 
    filter(unit_concept_name %in% c("thousand/ul", NA))

In [ ]:
Basophil_count %>% count(measurement_concept_id, standard_concept_name, sort = T)

Basophil_count %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(0, 0.5) + 
    theme(legend.position="bottom") + 
    guides(color=guide_legend(ncol=1)) 

In [ ]:
Basophil_count <- Basophil_count %>%
    filter(value_as_number >= 0  & value_as_number <= 0.5) %>% 
    mutate(unit_concept_name = "thousand/ul") %>% #easier dedup
    dedup_records() %>% dedup_median()

nrow(Basophil_count)
fivenum(Basophil_count$value_as_number)
Basophil_count_summ <- Basophil_count %>% group_by(person_id) %>% summarize(n=n())
fivenum(Basophil_count_summ$n)
nrow(Basophil_count_summ)

In [ ]:
write_to_bucket(Basophil_count, "bm_Basophil_count.csv")

In [ ]:
rm(Basophil_count_df, Basophil_count)
gc()

### Basophil fraction

In [ ]:
bm_concepts_key["BasoFra"]

In [ ]:
Basophil_fraction_df <- pull_lab(bm_concepts_key["BasoFra"], name = "BasoFra")                       
dim(Basophil_fraction_df)

In [ ]:
Basophil_fraction_df %>% count(unit_concept_name, sort=T)

In [ ]:
Basophil_fraction <- Basophil_fraction_df %>% 
    filter(unit_concept_name %in% c(NA, "percent", "percentage unit", "percent of white blood cells"))

In [ ]:
Basophil_fraction %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=log10(value_as_number), color = unit_concept_name)) + 
    geom_density() + xlim(-2, 2)

Basophil_fraction %>% count(measurement_concept_id, standard_concept_name, sort = T)

Basophil_fraction %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=log10(value_as_number), color = standard_concept_name)) + 
    geom_density() + xlim(-2, 2) + 
    theme(legend.position="bottom") + 
    guides(color=guide_legend(ncol=1)) 

In [ ]:
Basophil_fraction <- Basophil_fraction %>%
    filter(value_as_number >= 0.2  & value_as_number <= 2.1) %>%
    mutate(unit_concept_name = "percent") %>% #easier dedup
    dedup_records() %>% dedup_median()

nrow(Basophil_fraction)
fivenum(Basophil_fraction$value_as_number)
Basophil_fraction_summ <- Basophil_fraction %>% group_by(person_id) %>% summarize(n=n())
fivenum(Basophil_fraction_summ$n)
nrow(Basophil_fraction_summ)

In [ ]:
write_to_bucket(Basophil_fraction, "bm_Basophil_fraction.csv")

In [ ]:
rm(Basophil_fraction, Basophil_fraction_df)
gc()

### Eosinophil count

In [ ]:
Eosinophil_count_df <- pull_lab(bm_concepts_key["Eosinophil_count"], name = "Eosinophil_ct")
dim(Eosinophil_count_df)

In [ ]:
Eosinophil_count_df %>% count(unit_concept_name, sort=T)

Eosinophil_count_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + #+ xlim(0,20)
    scale_x_log10()

In [ ]:
Eosinophil_count <- Eosinophil_count_df %>%
    mutate(value_as_number = ifelse(is.na(unit_concept_name), value_as_number,
                                         ifelse(unit_concept_name %in% c("cells/microl", "cells/ul", "per microl"), 
                                                value_as_number/1000, value_as_number))) %>%
    mutate(unit_concept_name = ifelse(!is.na(unit_concept_name) & unit_concept_name %in% 
                                            c("thousand/cubic mm", "kelvin/microl", "kelvin/cubic mm",  
                                             "cells/microl", "cells/ul", "per microl", "thousand/microl" , 
                                              "cubic mm", "thousand"),  
                                      "thousand/ul", 
                                      unit_concept_name)) 

In [ ]:
Eosinophil_count %>% group_by(unit_concept_name) %>% filter(n() > 200) %>%
    ggplot(aes(x=log10(value_as_number), color = unit_concept_name)) + 
    geom_density() + xlim(-3, 3)

In [ ]:
Eosinophil_count <- Eosinophil_count %>%
    filter(unit_concept_name %in% c("thousand/ul", NA, "per cubic mm", "billion/l", "nl"))

In [ ]:
Eosinophil_count %>% count(measurement_concept_id, standard_concept_name, sort = T)

Eosinophil_count %>% group_by(standard_concept_name) %>% ## filter(n() > 100) %>%
    ggplot(aes(x=log10(value_as_number), color = standard_concept_name)) + 
    geom_density() + 
    theme(legend.position = "bottom") +
    guides(color = guide_legend(ncol=1)) + 
    xlim(log10(0.001), log10(30))

In [ ]:
Eosinophil_count <- Eosinophil_count %>%
    filter(value_as_number >= 0 & value_as_number <= 30) %>%
    dedup_records() %>% dedup_median()

nrow(Eosinophil_count)
fivenum(Eosinophil_count$value_as_number)
Eosinophil_count_summ <- Eosinophil_count %>% group_by(person_id) %>% summarize(n=n())
fivenum(Eosinophil_count_summ$n)
nrow(Eosinophil_count_summ)

In [ ]:
write_to_bucket(Eosinophil_count, "bm_Eosinophil_count.csv")

In [ ]:
rm(Eosinophil_count_df, Eosinophil_count)
gc()

### Eosinophil fraction

In [ ]:
bm_concepts_key["EosFra"]

In [ ]:
Eosinophil_fraction_df <- pull_lab(bm_concepts_key["EosFra"], name = "EosFra")
dim(Eosinophil_fraction_df)

In [ ]:
Eosinophil_fraction_df %>% count(unit_concept_name, sort=T)

In [ ]:
Eosinophil_fraction <- Eosinophil_fraction_df %>%
    filter(unit_concept_name %in% c(NA, "percent", "percentage unit", "percent of white blood cells"))

In [ ]:
Eosinophil_fraction %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density()  + xlim(0,20)

Eosinophil_fraction %>% count(measurement_concept_id, standard_concept_name, sort = T)

Eosinophil_fraction %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(0,20)

In [ ]:
Eosinophil_fraction <- Eosinophil_fraction %>%
    filter(value_as_number >= 0 & value_as_number <= 10) %>%
    mutate(unit_concept_name = "percent") %>% #easier dedup
    dedup_records() %>% dedup_median()

nrow(Eosinophil_fraction)
fivenum(Eosinophil_fraction$value_as_number)
Eosinophil_fraction_summ <- Eosinophil_fraction %>% group_by(person_id) %>% summarize(n=n())
fivenum(Eosinophil_fraction_summ$n)
nrow(Eosinophil_fraction_summ)

In [ ]:
write_to_bucket(Eosinophil_fraction, "bm_Eosinophil_fraction.csv")

In [ ]:
rm(Eosinophil_fraction, Eosinophil_fraction_df)
gc()

### Neutrophil count

In [ ]:
Neutrophil_count_df <- pull_lab(bm_concepts_key["Neutrophil_count"], name = "Neutrophil_ct")
dim(Neutrophil_count_df) 

In [ ]:
Neutrophil_count_df %>% count(unit_concept_name, sort=T)

Neutrophil_count_df %>% group_by(unit_concept_name) %>% filter(n() > 200) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + #+ xlim(0,20)
    scale_x_log10()

In [ ]:
Neutrophil_count <- Neutrophil_count_df %>%
    mutate(value_as_number = ifelse(is.na(unit_concept_name), value_as_number,
                                         ifelse(unit_concept_name %in% c("cells/microl", "cells/ul", "per microl"), 
                                                value_as_number/1000, value_as_number))) %>%
    mutate(unit_concept_name = ifelse(!is.na(unit_concept_name) & unit_concept_name %in% 
                                    c("thousand/cubic mm", "kelvin/microl",  "kelvin/cubic mm",
                                     "cells/microl", "cells/ul", "per microl", "thousand/microl", 
                                      "cubic mm"), #"billion/l","thousand", "x10(3)/mcl"
                              "thousand/ul", 
                               unit_concept_name)) %>%
    filter(!(!is.na(unit_concept_name) & unit_concept_name == "percent"))
    ###

In [ ]:
Neutrophil_count %>% group_by(unit_concept_name) %>% filter(n() > 200) %>%
    ggplot(aes(x=log10(value_as_number), color = unit_concept_name)) + 
    geom_density() + xlim(log10(0.1), log10(50)) + geom_vline(aes(xintercept=log10(30)))

In [ ]:
Neutrophil_count <- Neutrophil_count %>%
    filter(unit_concept_name %in% c("thousand/ul", "nl", "thousand", "billion/l", "x10(3)/mcl", NA)) %>%
    filter(!(is.na(unit_concept_name) & value_as_number > 30))

In [ ]:
Neutrophil_count %>% count(measurement_concept_id, standard_concept_name, sort = T)

Neutrophil_count %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=log10(value_as_number), color = standard_concept_name)) + 
    geom_density() + #+ xlim(0,20)
    theme(legend.position = "bottom") +
    guides(color = guide_legend(ncol=1)) + 
    xlim(log10(0.01), log10(100))

In [ ]:
Neutrophil_count <- Neutrophil_count %>%
    filter(value_as_number >= 0.1 & value_as_number <= 50) %>%
    mutate(unit_concept_name = "thousand/ul") %>% #easier dedup
    dedup_records() %>% dedup_median()

nrow(Neutrophil_count)
fivenum(Neutrophil_count$value_as_number)
Neutrophil_count_summ <- Neutrophil_count %>% group_by(person_id) %>% summarize(n=n())
fivenum(Neutrophil_count_summ$n)
nrow(Neutrophil_count_summ)

In [ ]:
write_to_bucket(Neutrophil_count, "bm_Neutrophil_count.csv")

In [ ]:
rm(Neutrophil_count, Neutrophil_count_df)
gc()

### Neutrophil fraction

In [ ]:
Neutrophil_fraction_df <- pull_lab(bm_concepts_key["NeutFra"], name = "NeutFra")    
dim(Neutrophil_fraction_df)

In [ ]:
Neutrophil_fraction_df %>% count(unit_concept_name, sort=T)

Neutrophil_fraction_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(0,100)

Neutrophil_fraction_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

In [ ]:
Neutrophil_fraction <- Neutrophil_fraction_df %>%
    filter(unit_concept_name %in% c("percent", NA, "percentage unit", "percent of white blood cells")) 

In [ ]:
Neutrophil_fraction %>% group_by(standard_concept_name) %>% 
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density(bw=.5)  + xlim(0,100) + geom_vline(aes(xintercept=85))

In [ ]:
Neutrophil_fraction <- Neutrophil_fraction %>%
    filter(value_as_number >= 0 & value_as_number <= 85) %>%
    dedup_records() %>% dedup_median()

nrow(Neutrophil_fraction)
fivenum(Neutrophil_fraction$value_as_number)
Neutrophil_fraction_summ <- Neutrophil_fraction %>% group_by(person_id) %>% summarize(n=n())
fivenum(Neutrophil_fraction_summ$n)
nrow(Neutrophil_fraction_summ)

In [ ]:
Neutrophil_fraction %>% group_by(standard_concept_name) %>% filter(n() > 1000) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density()  + xlim(0,100) + 
    theme(legend.position="bottom") + 
    guides(color=guide_legend(ncol=1))

In [ ]:
Neutrophil_fraction %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density()  + xlim(0,100)

In [ ]:
Neutrophil_fraction <- Neutrophil_fraction %>%
    filter(!(is.na(unit_concept_name) & value_as_number < 20)) # These are probably absolute counts (thousand/volume, etc)

In [ ]:
write_to_bucket(Neutrophil_fraction, "bm_Neutrophil_fraction.csv")

In [ ]:
rm(Neutrophil_fraction, Neutrophil_fraction_df)
gc()

### Lymphocyte Fraction

In [ ]:
bm_concepts_key["LymphFra"]

In [ ]:
LymphFra_df <- pull_lab(bm_concepts_key["LymphFra"], name = "LymphFra")
dim(LymphFra_df)

In [ ]:
LymphFra_df %>% count(unit_concept_name, sort=T)
LymphFra_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

In [ ]:
LymphFra <- LymphFra_df %>%
    filter(unit_concept_name %in% c(NA, "percent", "percentage unit", "percent of white blood cells"))

In [ ]:
LymphFra %>% group_by(standard_concept_name) %>% ##filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(0,75) +
    facet_wrap(~unit_concept_name) +
    theme(legend.position = "bottom") +
    guides(color = guide_legend(ncol=1))

Automated differ from manual counts across different unit types. 

In [ ]:
LymphFra <- LymphFra %>%
    filter(value_as_number >= 5 & value_as_number <= 75) %>%
    dedup_records() %>% dedup_median()
nrow(LymphFra)
fivenum(LymphFra$value_as_number)
LymphFra_summ <- LymphFra %>% group_by(person_id) %>% summarize(n=n())
fivenum(LymphFra_summ$n)
nrow(LymphFra_summ)

In [ ]:
write_to_bucket(LymphFra, "bm_LymphFra.csv")

In [ ]:
rm(LymphFra_df, LymphFra)
gc()

### Lymphocyte count

In [ ]:
bm_concepts_key["Lymphocyte_count"]

In [ ]:
Lymphocyte_count_df <- pull_lab(bm_concepts_key["Lymphocyte_count"], name = "Lymphocyte_count")
dim(Lymphocyte_count_df)

In [ ]:
Lymphocyte_count_df %>% count(unit_concept_name) %>% arrange(desc(n))

Lymphocyte_count_df %>% group_by(unit_concept_name) %>% filter(n() > 200) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(0.2,100)

In [ ]:
Lymphocyte_count <- Lymphocyte_count_df %>% 
    mutate(value_as_number = ifelse(is.na(unit_concept_name), value_as_number,
                                ifelse(unit_concept_name %in% c("per cumic mm",  "per microl", "cells/ul", 
                                                                "cells/microl", "cells/cubic mm"), 
                                                value_as_number/1000, value_as_number))) %>%
    mutate(unit_concept_name = ifelse(!is.na(unit_concept_name) & unit_concept_name %in% 
                                    c("thousand/cubic mm", "kelvin/microl",  "kelvin/cubic mm", "/mcl",
                                     "per cumic mm", "per microl", "cells/ul", "cells/microl", "cells/cubic mm", 
                                      "thousand/microl", "cubic mm", "x10(3)/mcl","thousand"), #"billion/l"
                              "thousand/ul", 
                               unit_concept_name)) %>%
    filter(!(!is.na(unit_concept_name) & unit_concept_name == "percent"))

In [ ]:
Lymphocyte_count %>% group_by(unit_concept_name) %>% filter(n() > 200) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(0.2,15)

In [ ]:
Lymphocyte_count <- Lymphocyte_count %>% 
    filter(unit_concept_name %in% c(NA, "billion/l", "nl", "thousand/ul"))

In [ ]:
Lymphocyte_count %>% group_by(standard_concept_name) %>% #filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(0.2,15) +
    theme(legend.position = "bottom") + 
    guides(color = guide_legend(ncol=1))

In [ ]:
Lymphocyte_count <- Lymphocyte_count %>%
    filter(value_as_number >= 0.2 & value_as_number <= 15) %>%
    mutate(unit_concept_name = "thousand/ul") %>% #easier dedup
    dedup_records() %>% dedup_median()
nrow(Lymphocyte_count)
fivenum(Lymphocyte_count$value_as_number)
Lymphocyte_count_summ <- Lymphocyte_count %>% group_by(person_id) %>% summarize(n=n())
fivenum(Lymphocyte_count_summ$n)
nrow(Lymphocyte_count_summ)

In [ ]:
write_to_bucket(Lymphocyte_count, "bm_Lymphocyte_count.csv")

In [ ]:
rm(Lymphocyte_count_df, Lymphocyte_count)
gc()

## Comprehensive Metabolic Panel - Others

### ALT

In [ ]:
ALT_df <- pull_lab(bm_concepts_key["ALT"], name = "ALT")
dim(ALT_df)

In [ ]:
ALT_df %>% count(unit_concept_name, sort=T)

ALT_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + scale_x_log10()

ALT_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

ALT_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + scale_x_log10() + 
    theme(legend.position ="bottom") + guides(color = guide_legend(ncol=1))

In [ ]:
ALT <- ALT_df %>%
    filter(measurement_concept_id != 3019056) %>%
    filter(unit_concept_name %in% c("unit/l", NA, "iu/l", "international unit/l", "nl", "u/l")) %>%
    filter(value_as_number > 0 & value_as_number <= 1e6)  %>% 
    mutate(unit_concept_name = "iu/l") %>% # easier dedup and all agree
    dedup_records() %>% dedup_median()

nrow(ALT)
fivenum(ALT$value_as_number)
ALT_summ <- ALT %>% group_by(person_id) %>% summarize(n=n())
fivenum(ALT_summ$n)
nrow(ALT_summ)

In [ ]:
write_to_bucket(ALT, "bm_ALT.csv")

In [ ]:
rm(ALT, ALT_df)
gc()

### AST (Aspartate Aminotransferase)

In [ ]:
AST_df <- pull_lab(bm_concepts_key["AST"], name = "AST")          
dim(AST_df)

In [ ]:
AST_df %>% count(unit_concept_name, sort=T)

AST_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=log10(value_as_number), color = unit_concept_name)) + 
    geom_density() #+ xlim(log10(0), log10(8000))

AST_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

AST_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=log10(value_as_number), color = standard_concept_name)) + 
    geom_density(show.legend=F) #+ xlim(log10(0), log10(8000))

In [ ]:
AST <- AST_df %>%
    filter(unit_concept_name %in% c("international unit/l", "iu/l",  "nl",  "unit/l", NA)) %>%
    filter(value_as_number >= 0 & value_as_number <= 8000) %>%
    mutate(unit_concept_name = "iu/l") %>% # easier dedup and all agree
    dedup_records()  %>% dedup_median()

nrow(AST)
fivenum(AST$value_as_number)
AST_summ <- AST %>% group_by(person_id) %>% summarize(n=n())
fivenum(AST_summ$n)
nrow(AST_summ)

In [ ]:
write_to_bucket(AST, "bm_AST.csv")

In [ ]:
rm(AST, AST_df)
gc()

### Alkaline Phosphatase (ALP / PA)

In [ ]:
Alkaline_Phosphatase_df <- pull_lab(bm_concepts_key["PA"], name = "alkaline_phosphatase")
dim(Alkaline_Phosphatase_df)

In [ ]:
Alkaline_Phosphatase_df %>% count(unit_concept_name, sort=T)

Alkaline_Phosphatase_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=log10(value_as_number), color = unit_concept_name)) + 
    geom_density() + xlim(log10(5), log10(5000))

Alkaline_Phosphatase_df %>% count(measurement_concept_id, standard_concept_name, sort=T)

Alkaline_Phosphatase_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=log10(value_as_number), color = standard_concept_name)) + 
    geom_density()  + xlim(log10(5), log10(5000)) + 
    theme(legend.position="bottom") + 
    guides(color=guide_legend(ncol=1))

In [ ]:
Alkaline_Phosphatase <- Alkaline_Phosphatase_df %>%
    filter(!(measurement_concept_id %in% c(4154344, 4151549))) %>%
    filter(unit_concept_name %in% c("iu/l",  "nl", "international unit/l", "u/l", "unit/l",  NA)) %>%
    mutate(unit_concept_name = "iu/l") %>% #easier dedup and good agreement
    filter(value_as_number >= 5 & value_as_number <= 5000) %>%
    dedup_records()  %>% dedup_median()

nrow(Alkaline_Phosphatase)
fivenum(Alkaline_Phosphatase$value_as_number)
Alkaline_Phosphatase_summ <- Alkaline_Phosphatase %>% group_by(person_id) %>% summarize(n=n())
fivenum(Alkaline_Phosphatase_summ$n)
nrow(Alkaline_Phosphatase_summ)

In [ ]:
write_to_bucket(Alkaline_Phosphatase, "bm_Alkaline_Phosphatase.csv")

In [ ]:
rm(Alkaline_Phosphatase, Alkaline_Phosphatase_df)
gc()

### Albumin (blood, serum, plasma)

In [ ]:
Albumin_BSP_df <- pull_lab(bm_concepts_key["Albumin"], name = "albumin_bsp")
dim(Albumin_BSP_df)

In [ ]:
Albumin_BSP_df %>% count(unit_concept_name, sort=T)

Albumin_BSP_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(0.8, 60)

Albumin_BSP_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

Albumin_BSP_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(0.8, 60) + 
    theme(legend.position="bottom") + guides(color = guide_legend(ncol=1))

In [ ]:
Albumin_BSP_df %>% filter(measurement_concept_id == 4017497) %>% 
    count(value_as_number, sort=T) %>% head(5)

These values tend to be the upper or lower limits of reference ranges and are probably not the true value. 

In [ ]:
Albumin_BSP <- Albumin_BSP_df %>%
    filter(!(measurement_concept_id %in% c(4017497, 3043798))) %>%
    #don't allow concept ids not corresponding to mass/volume
    filter(!(measurement_concept_id %in% c(3020509, 3046948, 3007424,3035472, 40757478, 43055432))) %>%
    filter(unit_concept_name %in% c("g/l", NA, "g/dl", "g/dl calculated")) %>%
    mutate(value_as_number = ifelse(!is.na(unit_concept_name) & unit_concept_name == "g/l", 
                                    value_as_number/10, value_as_number),
          unit_concept_name = ifelse(!is.na(unit_concept_name) & unit_concept_name == "g/l", 
                                     "g/dl", unit_concept_name)) %>%
    filter(value_as_number >= 0.8 & value_as_number <= 10) %>% 
    dedup_records() %>% dedup_median()

In [ ]:
nrow(Albumin_BSP)
fivenum(Albumin_BSP$value_as_number)
Albumin_BSP_summ <- Albumin_BSP %>% group_by(person_id) %>% summarize(n=n())
fivenum(Albumin_BSP_summ$n)
nrow(Albumin_BSP_summ)

In [ ]:
write_to_bucket(Albumin_BSP, "bm_Albumin_BSP.csv")

In [ ]:
rm(Albumin_BSP, Albumin_BSP_df)
gc()

### Bicarbonate

In [ ]:
bm_concepts_key["Bicarbonate"]

In [ ]:
Bicarbonate_bsp_df <- pull_lab(bm_concepts_key["Bicarbonate"], name = "bicarb", read_existing = F)
dim(Bicarbonate_bsp_df)

In [ ]:
Bicarbonate_bsp_df %>% count(unit_concept_name, sort=T)

Bicarbonate_bsp_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=log10(value_as_number), color = unit_concept_name)) + 
    geom_density(bw = 0.01) + #scale_x_log10() + 
    xlim(log10(5), log10(100))

Bicarbonate_bsp_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

In [ ]:
Bicarbonate_bsp <- Bicarbonate_bsp_df %>%
    filter(unit_concept_name %in% c("mmole/l", NA, "mequivalent/l", "microequivalent/l", "meq/l",
                                    "mequivalent/ml", "mmol/l"))

Since there are so many concepts, split them into Bicarbonate vs CO2 concepts and analyze separately + compare

In [ ]:
#Bicarbonate only 
Bicarbonate_bsp %>% group_by(standard_concept_name) %>% 
    filter(grepl("Bicarbonate", standard_concept_name)) %>%
    ggplot(aes(x=log10(value_as_number), color = standard_concept_name)) + 
    geom_density(bw=.01) + xlim(log10(5), log10(55)) + 
    theme(legend.position = "bottom") +
    guides(color = guide_legend(ncol=1))

In [ ]:
#CO2
Bicarbonate_bsp %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    filter(!grepl("Bicarbonate", standard_concept_name)) %>%
    ggplot(aes(x=log10(value_as_number), color = standard_concept_name)) + 
    geom_density(bw = 0.01) + xlim(log10(5), log10(100)) + 
    theme(legend.position = "bottom") +
    guides(color = guide_legend(ncol=1)) 

In [ ]:
bicarb_vs_co2_summ <- Bicarbonate_bsp %>%
    filter(!(measurement_concept_id %in% c(3018225))) %>%
    filter(value_as_number >= 5 & value_as_number <= 55) %>%
    mutate(type = ifelse(grepl("Bicarb", standard_concept_name), "Bicarb", "CO2")) %>%
    group_by(type) %>%
    summarize(median = median(value_as_number),
             mean = mean(value_as_number))

bicarb_vs_co2_summ

Bicarb vs. CO2 are very consistent, as expected

In [ ]:
Bicarbonate_bsp <- Bicarbonate_bsp %>%
    filter(!(measurement_concept_id %in% c(3018225))) %>%
    filter(value_as_number >= 5 & value_as_number <= 55) %>% 
    dedup_records() %>% dedup_median()

nrow(Bicarbonate_bsp)
fivenum(Bicarbonate_bsp$value_as_number)
Bicarbonate_bsp_summ <- Bicarbonate_bsp %>% group_by(person_id) %>% summarize(n=n())
fivenum(Bicarbonate_bsp_summ$n)
nrow(Bicarbonate_bsp_summ)

In [ ]:
write_to_bucket(Bicarbonate_bsp, "bm_Bicarbonate_bsp.csv")

In [ ]:
rm(Bicarbonate_bsp, Bicarbonate_bsp_df)

### Sodium in blood, serum, plasma

In [ ]:
Sodium_bsp_df <- pull_lab(bm_concepts_key["Sodium_bsp"], name = "sodium_bsp")
dim(Sodium_bsp_df)

In [ ]:
Sodium_bsp_df %>% count(unit_concept_name, sort=T)

Sodium_bsp_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density(bw = 0.3) + xlim(110, 170)

Sodium_bsp_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

Sodium_bsp_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density(bw = 0.3) + xlim(110, 170)

In [ ]:
Sodium_bsp <- Sodium_bsp_df %>%
    filter(unit_concept_name %in% c("mmole/l", NA, "microequivalent/l", "equivalent/l", "mequivalent/l", "mmol/l")) %>%
    mutate(unit_concept_name = "mmole/l") %>% #easier dedup, good agreement
    filter(value_as_number >= 95 & value_as_number <= 225) %>% 
    dedup_records() %>% dedup_median()
nrow(Sodium_bsp)
fivenum(Sodium_bsp$value_as_number)
Sodium_bsp_summ <- Sodium_bsp %>% group_by(person_id) %>% summarize(n=n())
fivenum(Sodium_bsp_summ$n)
nrow(Sodium_bsp_summ)

In [ ]:
write_to_bucket(Sodium_bsp, "bm_Sodium_bsp.csv")

In [ ]:
rm(Sodium_bsp, Sodium_bsp_df)

### Magnesium (Mg) in blood, serum, plasma

In [ ]:
Magnesium_bsp_df <- pull_lab(bm_concepts_key["Mg"], name = "Mg_bsp")
dim(Magnesium_bsp_df)

In [ ]:
Magnesium_bsp_df %>% count(unit_concept_name, sort=T)

Magnesium_bsp_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density(bw=.05) + xlim(0.5, 4)

Magnesium_bsp_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

Magnesium_bsp_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density(bw=.05) + xlim(0.5, 4)

In [ ]:
Magnesium_bsp <- Magnesium_bsp_df %>%
    filter(unit_concept_name %in% c("meq/l", "mg/dl", "mequivalent/l", "mg/ml", NA)) %>%
    mutate(value_as_number = ifelse(!is.na(unit_concept_name) & unit_concept_name %in% c("meq/l", "mequivalent/l"), 
                                   1.2153*value_as_number, value_as_number)) %>%
    mutate(unit_concept_name = "mg/dl") %>%
    filter(value_as_number >= 0.5 & value_as_number <= 8) %>% 
    dedup_records() %>% dedup_median()

nrow(Magnesium_bsp)
fivenum(Magnesium_bsp$value_as_number)
Magnesium_bsp_summ <- Magnesium_bsp %>% group_by(person_id) %>% summarize(n=n())
fivenum(Magnesium_bsp_summ$n)
nrow(Magnesium_bsp_summ)

In [ ]:
write_to_bucket(Magnesium_bsp, "bm_Magnesium_bsp.csv")

In [ ]:
rm(Magnesium_bsp, Magnesium_bsp_df)
gc()

### Calcium

In [ ]:
Calcium_bsp_df <- pull_lab(bm_concepts_key["Calcium_bsp"], name = "calcium_bsp")
dim(Calcium_bsp_df)

In [ ]:
Calcium_bsp_df %>% count(unit_concept_name, sort=T)

Calcium_bsp_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + 
    #scale_x_log10() + 
    xlim(0,20)

Calcium_bsp_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

Calcium_bsp_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + ###scale_x_log10() + 
    xlim(6.5,12) + 
    theme(legend.position ="bottom") + guides(color = guide_legend(ncol=1))

In [ ]:
Calcium_bsp <- Calcium_bsp_df %>%
    filter(!(measurement_concept_id %in% 
             c(46272910, 3015377, 44805588, 3020059, 3005162, 3025742, 3026798, 4307178, 3007393))) %>%
    filter(unit_concept_name %in% c("mg/dl", "mg/ml", NA)) %>%
    filter(value_as_number >= 6.5 & value_as_number <= 18)  %>% 
    dedup_records() %>% dedup_median()

nrow(Calcium_bsp)
fivenum(Calcium_bsp$value_as_number)
Calcium_bsp_summ <- Calcium_bsp %>% group_by(person_id) %>% summarize(n=n())
fivenum(Calcium_bsp_summ$n)
nrow(Calcium_bsp_summ)

In [ ]:
write_to_bucket(Calcium_bsp, "bm_Calcium_bsp.csv")

In [ ]:
rm(Calcium_bsp, Calcium_bsp_df)

### Chloride in blood, serum, plasma

In [ ]:
Chloride_bsp_df <- pull_lab(bm_concepts_key["Cl"], name = "chloride_bsp")                             
dim(Chloride_bsp_df)

In [ ]:
Chloride_bsp_df %>% count(unit_concept_name, sort=T)

Chloride_bsp_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + 
    xlim(75,125) + 
    theme(legend.position ="bottom") 

Chloride_bsp_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

Chloride_bsp_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + 
    #scale_x_log10() + 
    xlim(75,125) + 
    theme(legend.position ="bottom") + guides(color = guide_legend(ncol=1))

In [ ]:
Chloride_bsp <- Chloride_bsp_df %>%
    filter(unit_concept_name %in% c("mmole/l", NA, "mequivalent/l", "microequivalent/l", "mmole",
                                    "equivalent/l", "meq/l", "mmol/l")) %>%
    filter(value_as_number >= 60 & value_as_number <= 150) %>% 
    dedup_records()  %>% dedup_median()
nrow(Chloride_bsp)
fivenum(Chloride_bsp$value_as_number)
Chloride_bsp_summ <- Chloride_bsp %>% group_by(person_id) %>% summarize(n=n())
fivenum(Chloride_bsp_summ$n)
nrow(Chloride_bsp_summ)

In [ ]:
write_to_bucket(Chloride_bsp, "bm_Chloride_bsp.csv")

In [ ]:
rm(Chloride_bsp, Chloride_bsp_df)

### Potassium (K)

In [ ]:
Potassium_df <- pull_lab(bm_concepts_key["Potassium"], name = "potassium")
dim(Potassium_df)

In [ ]:
Potassium_df %>% count(unit_concept_name, sort=T)

Potassium_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density(bw=0.1) + xlim(2, 8.5) + 
    theme(legend.position="bottom")

In [ ]:
Potassium <- Potassium_df %>% 
    filter(unit_concept_name %in% c("mmole/l", NA, "mequivalent/l", "microequivalent/l", "equivalent/l", "mmol/l")) 

In [ ]:
Potassium %>% count(measurement_concept_id, standard_concept_name, sort = T)

Potassium %>% group_by(standard_concept_name) %>% filter(n() > 1000) %>% 
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density(bw=0.1)  + xlim(2, 8.5) + 
    theme(legend.position="bottom") + 
    guides(color=guide_legend(ncol=T))

In [ ]:
Potassium <- Potassium %>%
    filter(!(measurement_concept_id %in% c(4154489, 21490733))) %>% # strange bimodal
    filter(value_as_number >= 2 & value_as_number <= 8.5) %>% 
    dedup_records() %>% dedup_median()
nrow(Potassium)
fivenum(Potassium$value_as_number)
Potassium_summ <- Potassium %>% group_by(person_id) %>% summarize(n=n())
fivenum(Potassium_summ$n)
nrow(Potassium_summ)

In [ ]:
write_to_bucket(Potassium, "bm_Potassium.csv")

In [ ]:
rm(Potassium, Potassium_df)
gc()

### BUN (Blood Urea Nitrogen)

In [ ]:
BUN_df <- pull_lab(bm_concepts_key["BUN"], name = "BUN")
dim(BUN_df)

In [ ]:
BUN_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

BUN_df %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(0.2, 100) + #scale_x_log10() +
    theme(legend.position ="bottom") + guides(color = guide_legend(ncol=1))

In [ ]:
BUN <- BUN_df %>% 
    filter(!(measurement_concept_id %in% #mass ratio, moles/vol, urine
             c(3018311, 4112223, 3024641, 3046485, 4020122, 4017362, 40762632, 3053280))) 

In [ ]:
BUN %>% count(unit_concept_name, sort=T)

BUN %>% group_by(unit_concept_name) %>% filter(n() > 1000) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density(bw=.5) + xlim(0.2, 50) #+ scale_x_log10()

In [ ]:
BUN <- BUN %>% filter(unit_concept_name %in% c("mg/dl", "mg/ml" , NA)) %>%
    mutate(unit_concept_name = "mg/dl") %>%    
    filter(value_as_number >= 0.2 & value_as_number <= 400) %>% 
    dedup_records() %>% dedup_median()
nrow(BUN)
fivenum(BUN$value_as_number)
BUN_summ <- BUN %>% group_by(person_id) %>% summarize(n=n())
fivenum(BUN_summ$n)
nrow(BUN_summ)

In [ ]:
write_to_bucket(BUN, "bm_BUN.csv")

In [ ]:
rm(BUN, BUN_df)
gc()

## Other Labs

### CRP

In [ ]:
CRP_df <- pull_lab(bm_concepts_key["CRP"], name = "CRP")
dim(CRP_df)

In [ ]:
CRP_df %>% count(unit_concept_name, sort=T)

CRP_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=log(value_as_number, 10), color = unit_concept_name)) + 
    geom_density()

In [ ]:
CRP <- CRP_df %>%
    mutate(value_as_number = ifelse(!is.na(unit_concept_name) & unit_concept_name == "mg/l", 
                                    value_as_number/10, value_as_number)) %>%
    mutate(unit_concept_name = ifelse(!is.na(unit_concept_name) & unit_concept_name == "mg/l", 
                                      "mg/dl", unit_concept_name)) %>%
    filter(unit_concept_name %in% c("mg/dl", NA))

In [ ]:
CRP %>% count(measurement_concept_id, standard_concept_name, sort = T)

CRP %>% ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() +
    facet_grid(standard_concept_name~.)+
    theme(legend.position = "bottom") +
    scale_x_log10()

In [ ]:
CRP <- CRP %>% 
    mutate(value_as_number = ifelse(is.na(unit_concept_name) & measurement_concept_id == 3010156, 
                                    value_as_number/10, value_as_number)) %>%
    filter(value_as_number >= 0.1 & value_as_number <= 1000)
nrow(CRP)

In [ ]:
CRP <- CRP %>% dedup_records() %>% dedup_median()

nrow(CRP)
fivenum(CRP$value_as_number)
CRP_summ <- CRP %>% group_by(person_id) %>% summarize(n=n())
fivenum(CRP_summ$n)
nrow(CRP_summ)

In [ ]:
write_to_bucket(CRP, "bm_CRP.csv")

In [ ]:
rm(CRP_df, CRP)
gc()

### Serum Creatinine and derived EGFR

In [ ]:
bm_concepts_key["Serum_Creatinine"]
serum_creatinine_df <- pull_lab(bm_concepts_key["Serum_Creatinine"], name = "serum_creatinine")
dim(serum_creatinine_df)

In [ ]:
serum_creatinine_df %>% count(unit_concept_name, sort=T)

# serum_creatinine_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
#     ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
#     geom_density() +
#     xlim(0.1, 5)

serum_creatinine_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

serum_creatinine_df %>% group_by(standard_concept_name, unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    facet_wrap(~unit_concept_name) +
    geom_density() +
    xlim(0.1, 5) +
    theme(legend.position = "bottom") + guides(color = guide_legend(ncol=1))

In [ ]:
serum_creatinine <- serum_creatinine_df %>%
    filter(value_as_number >= 0.2 & value_as_number <= 10) %>%
    filter(!measurement_concept_id %in% c(3033837)) %>% 
    filter(unit_concept_name %in% c("mg/dl", "micromole/l", "mg/ml", NA)) %>%
    dedup_records() %>% dedup_median()

nrow(serum_creatinine)
fivenum(serum_creatinine$value_as_number)
serum_creatinine_summ <- serum_creatinine %>% group_by(person_id) %>% summarize(n=n())
fivenum(serum_creatinine_summ$n)
nrow(serum_creatinine_summ)

In [ ]:
demog <- read_cols("covariates_wide_all_participants.csv", skip_copy=TRUE, 
                   select = c("person_id", "date_of_birth", "sex_at_birth", "gender")) %>% 
    mutate(sex = coalesce(sex_at_birth, gender)) %>%
    select(-sex_at_birth, -gender)

In [ ]:
# calculate eGFR. Need age and sex. to calculate age, need DOB. 
# Define eGFR formula

compute_egfr <- function(creat_blood, age, sex){
  sex <- tolower(sex)
  if(is.na(sex) | !(sex %in% c("male", "female"))) {return(NA)}
  
  kappa_egfr <- NA
  alpha_egfr <- NA
  
  if(sex == "female"){
    #kappa_egfr <- 61.9
    kappa_egfr <- 0.7
    #alpha_egfr  <- -0.329
    alpha_egfr <- 0.241
  }
  else if(sex == "male"){
    #kappa_egfr <- 79.6
    kappa_egfr <- 0.9
    #alpha_egfr <- -0.411
    alpha_egfr <- -0.302
  }
  
  if(is.na(kappa_egfr) || is.na(alpha_egfr)) return(NA)
  
  egfr <- 142*pmin(creat_blood/kappa_egfr,1)^alpha_egfr * pmax(creat_blood/kappa_egfr, 1)^(-1.2)*0.9938^age
  egfr <- egfr * ifelse(sex=="female", 1.012, 1)
  return(egfr)
}

In [ ]:
#Add demographics to creatinine
tic()
eGFR_creatinine <- serum_creatinine %>% 
    left_join(demog) %>%
    mutate(age_egfr = decimal_date(as.Date(measurement_date)) - decimal_date(as.Date(date_of_birth)))

#Compute EGFR
eGFR_creatinine$eGFR <-
  mapply(compute_egfr, eGFR_creatinine$value_as_number, eGFR_creatinine$age_egfr, eGFR_creatinine$sex) 
toc()

fivenum(eGFR_creatinine$eGFR) %>% signif(4)

In [ ]:
eGFR_creatinine <- eGFR_creatinine %>% mutate(eGFR = pmin(eGFR, 200)) # set a max of 200 for estimated gfr

In [ ]:
eGFR_creatinine %>% filter(is.na(mult_type_summary_flag)) %>%
    group_by(standard_concept_name) %>% filter(n() >  10) %>%
    ggplot(aes(x=eGFR, color = standard_concept_name)) + geom_density() + 
    theme(legend.position = "bottom") + guides(color = guide_legend(ncol=1))

people with ESRD may get their creatinine measured very often

In [ ]:
fivenum(eGFR_creatinine$eGFR)
eGFR_summ <- eGFR_creatinine %>% group_by(person_id) %>% summarize(n=n())
fivenum(eGFR_summ$n)

In [ ]:
eGFR_creatinine <- eGFR_creatinine %>% 
    distinct(person_id, measurement_date, eGFR, Creatinine = value_as_number, visit_occurrence_concept_name, 
           measurement_concept_id, standard_concept_name, unit_concept_name, summarized_from_n, 
           summarized_concept_ids, mult_type_summary_flag)

In [ ]:
write_to_bucket(eGFR_creatinine, "bm_eGFR_creatinine.csv")

In [ ]:
rm(eGFR_creatinine, serum_creatinine, serum_creatinine_df)
gc()

### Albumin in urine, Creatinine in urine, and UACR

#### Extract urine albumin separately

In [ ]:
Albumin_urine_df <- pull_lab(bm_concepts_key["Albumin_urine"], name = "albumin_urine")
dim(Albumin_urine_df)

In [ ]:
Albumin_urine_df %>% count(unit_concept_name, sort=T)

In [ ]:
Albumin_urine <- Albumin_urine_df %>% filter(unit_concept_name %in% c("mg/dl","mg/l",NA,"g/dl","microg/ml","mg/ml"))

In [ ]:
Albumin_urine %>% count(measurement_concept_id, standard_concept_name, sort = T)

In [ ]:
Albumin_urine <- Albumin_urine %>% filter(measurement_concept_id %in% c(3000034, 3012516, 3039775))
#This data is messy, not worth trying to clean rare ones

In [ ]:
Albumin_urine %>% group_by(unit_concept_name) %>% filter(n() > 250) %>%
    ggplot(aes(x=log10(value_as_number+0.01), color = unit_concept_name)) + 
    geom_density() + xlim(log10(0.01), 5) #log10(60))

In [ ]:
Albumin_urine <- Albumin_urine %>%
    filter(unit_concept_name %in% c("mg/dl",  "mg/l" , NA, "g/dl", "microg/ml")) %>%
    mutate(value_as_number = ifelse(is.na(unit_concept_name), value_as_number, 
                                 ifelse(unit_concept_name %in% c("microg/ml", "mg/l"), value_as_number*0.1, 
                                         value_as_number)),
           unit_concept_name = ifelse(unit_concept_name %in% c("microg/ml", "mg/l"), "mg/dl converted", 
                                         unit_concept_name)
          )

In [ ]:
Albumin_urine %>% group_by(unit_concept_name) %>% #filter(n() > 250) %>%
    ggplot(aes(x=log10(value_as_number+0.01), color = unit_concept_name)) + 
    geom_density() + xlim(log10(0.01), 5) #log10(60))

In [ ]:
Albumin_urine <- Albumin_urine %>% 
    mutate(value_as_number = ifelse(is.na(unit_concept_name) & measurement_concept_id == 3012516,
                                   value_as_number/10, value_as_number),
          unit_concept_name = ifelse(is.na(unit_concept_name) & measurement_concept_id == 3012516, 
                                     "mg/dl conv from NA", unit_concept_name))

In [ ]:
Albumin_urine %>% group_by(standard_concept_name, unit_concept_name, visit_occurrence_concept_name) %>% 
    filter(n() > 250) %>%
    ggplot(aes(x=log10(value_as_number + 0.01), color = standard_concept_name)) + 
    geom_density() + 
    facet_wrap(visit_occurrence_concept_name~unit_concept_name, scales="free_y") +
    theme(legend.position ="bottom") + guides(color = guide_legend(ncol=1)) +
    xlim(-1, 3)

In [ ]:
Albumin_urine <- Albumin_urine %>%
    filter(value_as_number >= 0.1 & value_as_number <= 1000) %>%
    dedup_records() %>% dedup_median() %>% 
    #Don't allow summaries across different units/concept ids
    filter(is.na(mult_type_summary_flag))

nrow(Albumin_urine)
fivenum(Albumin_urine$value_as_number)
Albumin_urine_summ <- Albumin_urine %>% group_by(person_id) %>% summarize(n=n())
fivenum(Albumin_urine_summ$n)

In [ ]:
write_to_bucket(Albumin_urine, "bm_Albumin_urine.csv")

In [ ]:
poss_UACR_from_Albumin <- Albumin_urine_df %>% filter(unit_concept_name %in% c("mg/g of creatinine"," mg/g"))

In [ ]:
poss_UACR_from_Albumin %>% 
    ggplot(aes(x=log10(value_as_number), color = factor(standard_concept_name))) + 
    geom_density() + 
    theme(legend.position ="bottom") + guides(color = guide_legend(ncol=1))

#### Extract urine creatinine separately

In [ ]:
Creatinine_urine_df <- pull_lab(bm_concepts_key["Creatinine_urine"], name = "creatinine_urine")
dim(Creatinine_urine_df)

In [ ]:
Creatinine_urine_df %>% count(unit_concept_name, sort=T)
Creatinine_urine_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

In [ ]:
Creatinine_urine <- Creatinine_urine_df %>%
    filter(!(measurement_concept_id %in% c(3001349, 4230629, 4133756))) %>% 
    filter(unit_concept_name %in% c("mg/dl", NA, "mg/l")) %>%

In [ ]:
Creatinine_urine %>% 
    ggplot(aes(x=log10(value_as_number+0.01), color = unit_concept_name)) + 
    geom_density() + xlim(-2, 4)

In [ ]:
Creatinine_urine <- Creatinine_urine %>%
    mutate(value_as_number = ifelse(is.na(unit_concept_name), value_as_number,
                                    ifelse(unit_concept_name == "mg/l", value_as_number*0.1, value_as_number)),
           unit_concept_name  = ifelse(unit_concept_name == "mg/l", "mg/dl", unit_concept_name))

In [ ]:
Creatinine_urine %>% group_by(standard_concept_name) %>% #filter(n() > 100) %>%
    ggplot(aes(x=log10(value_as_number), color = standard_concept_name)) + 
    geom_density() + 
    xlim(log10(30), log10(320)) + 
    theme(legend.position = "bottom") + 
    guides(color = guide_legend(ncol=1)) + 
    facet_wrap(~unit_concept_name, scales="free_y")

In [ ]:
Creatinine_urine <- Creatinine_urine %>%
    filter(value_as_number >= 30 & value_as_number <= 320) %>% 
    dedup_records() %>% dedup_median() %>% 
    filter(summarized_from_n == 1)

nrow(Creatinine_urine)
fivenum(Creatinine_urine$value_as_number)
Creatinine_urine_summ <- Creatinine_urine %>% group_by(person_id) %>% summarize(n=n())
fivenum(Creatinine_urine_summ$n)
nrow(Creatinine_urine_summ)

In [ ]:
write_to_bucket(Creatinine_urine, "bm_Creatinine_urine.csv")

#### Join urine albumin and creatinine to calculate UACR

In [ ]:
uacr_calculated <- Albumin_urine %>% 
    distinct(person_id, measurement_date, 
             urine_albumin = value_as_number, 
             visit_occurrence_concept_name_albumin = visit_occurrence_concept_name,
             unit_concept_name_albumin = unit_concept_name, 
             standard_concept_name_albumin =  standard_concept_name, 
             measurement_concept_id_albumin = measurement_concept_id, 
             summarized_concept_ids_albumin = summarized_concept_ids, 
             mult_type_summary_flag_albumin = mult_type_summary_flag
            ) %>%
    inner_join(
        Creatinine_urine %>% 
             distinct(person_id, measurement_date, 
                 urine_creatinine = value_as_number, 
                 visit_occurrence_concept_name_creatinine = visit_occurrence_concept_name,
                 unit_concept_name_creatinine = unit_concept_name, 
                 standard_concept_name_creatinine =  standard_concept_name, 
                 measurement_concept_id_creatinine = measurement_concept_id, 
                 summarized_concept_ids_creatinine = summarized_concept_ids, 
                 mult_type_summary_flag_creatinine = mult_type_summary_flag)
    ) %>%
    mutate(UACR_calculated = signif(1000*urine_albumin/urine_creatinine, 3)) %>%
    filter(UACR_calculated >= 0.1 & UACR_calculated <= 20000)

fivenum(uacr_calculated$UACR_calculated)
uacr_calculated_summ <- uacr_calculated %>% group_by(person_id) %>% summarize(n=n())
fivenum(uacr_calculated_summ$n)
nrow(uacr_calculated_summ)

In [ ]:
uacr_calculated %>% group_by(measurement_concept_id_albumin, measurement_concept_id_creatinine) %>%
    filter(n() > 100) %>%
    ggplot(aes(x=UACR_calculated, color=factor(measurement_concept_id_creatinine))) + 
    geom_density(bw=.05) + 
    geom_vline(xintercept = 30) + geom_vline(xintercept = 300) + scale_x_log10() +
    facet_wrap(measurement_concept_id_albumin~., scales="free_y") + 
    theme(legend.position = "bottom") + guides(color = guide_legend(ncol=1))

uacr_calculated %>% group_by(measurement_concept_id_albumin, measurement_concept_id_creatinine) %>%
    filter(n() > 100) %>%
    ggplot(aes(x=UACR_calculated, color=factor(measurement_concept_id_albumin))) + 
    geom_density(bw=.05) + 
    geom_vline(xintercept = 30) + geom_vline(xintercept = 300) + scale_x_log10() +
    facet_wrap(measurement_concept_id_creatinine~., scales="free_y") + 
    theme(legend.position = "bottom") + guides(color = guide_legend(ncol=1))

In [ ]:
uacr_calculated %>% group_by(measurement_concept_id_albumin, measurement_concept_id_creatinine) %>%
    filter(n() > 100) %>%
    ggplot(aes(x=UACR_calculated, color=factor(measurement_concept_id_creatinine), 
               linetype=factor(measurement_concept_id_albumin))) + 
    geom_density(bw=.05) + 
    geom_vline(xintercept = 30) + geom_vline(xintercept = 300) + scale_x_log10() 

#### Pull UACR direct

In [ ]:
# UACR_df <- pull_lab(bm_concepts_key["UACR"], name = "UACR")
# dim(UACR_df)

We need some more details for UACR that are not in the standard query. It is difficult to clean the units on this one due to high missingness and a very broad range of allowable values spanning multiple orders of magnitude. 

In [ ]:
bm_concepts_key["UACR"]
name = "UACR2"
concept_ids_string = bm_concepts_key["UACR"]
read_existing=F

biomarker_path <- file.path(
      Sys.getenv("WORKSPACE_BUCKET"),
      "bq_exports",
      Sys.getenv("OWNER_EMAIL"),
      paste0("measurement_", name, "_*.csv") )

system(paste0("gsutil rm ", biomarker_path), intern=T)


In [ ]:
if (!read_existing) {
tic(paste("Pulling data for ", name))

biomarker_sql <- paste("
    SELECT DISTINCT
        measurement.person_id,
        measurement.measurement_concept_id,
        measurement.measurement_datetime as measurement_date,
        measurement.value_as_number,        
        measurement.range_low,
        measurement.range_high,
        measurement.unit_source_value,
        measurement.value_source_value, 
        m_standard_concept.concept_name as standard_concept_name,
        m_unit.concept_name as unit_concept_name,
        m_visit.concept_name as visit_occurrence_concept_name, 
    m_type.concept_name as measurement_type_concept_name,
    m_operator.concept_name as operator_concept_name,
        measurement.measurement_source_concept_id,
        m_source_concept.concept_name as source_concept_name,
        m_source_concept.concept_code as source_concept_code,
        m_source_concept.vocabulary_id as source_vocabulary

    FROM
        ( SELECT
            * 
        FROM
            `measurement` measurement 
        WHERE
            (
                measurement_concept_id IN  (
                    SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `cb_criteria` c 
                    JOIN
                        (
                            select
                                cast(cr.id as string) as id 
                            FROM
                                `cb_criteria` cr 
                            WHERE
                                concept_id IN ( ",
                                    concept_ids_string,
                               " ) 
                                AND full_text LIKE '%_rank1]%'
                        ) a 
                            ON (
                                c.path LIKE CONCAT('%.',
                            a.id,
                            '.%') 
                            OR c.path LIKE CONCAT('%.',
                            a.id) 
                            OR c.path LIKE CONCAT(a.id,
                            '.%') 
                            OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1
                        )
                )
            ) measurement 
        LEFT JOIN
            `concept` m_standard_concept 
                ON measurement.measurement_concept_id = m_standard_concept.concept_id 
        LEFT JOIN
            `concept` m_type 
                ON measurement.measurement_type_concept_id = m_type.concept_id 
        LEFT JOIN
            `concept` m_operator 
                ON measurement.operator_concept_id = m_operator.concept_id 

        LEFT JOIN
            `concept` m_unit 
                ON measurement.unit_concept_id = m_unit.concept_id 
        LEFT JOIn
            `visit_occurrence` v 
                ON measurement.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` m_visit 
                ON v.visit_concept_id = m_visit.concept_id
        LEFT JOIN
            `concept` m_source_concept 
                ON measurement.measurement_source_concept_id = m_source_concept.concept_id", sep="")

bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), biomarker_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  biomarker_path,
  destination_format = "CSV")

} else { tic(paste("Reading existing pulled data for ", name))}

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), unit_concept_name = col_character(), 
                    visit_occurrence_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          #message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))}

biomarker_df <- suppressWarnings(read_bq_export_from_workspace_bucket(biomarker_path)) %>%
   mutate(measurement_date = as.Date(measurement_date)) ##%>%
   ##mutate(unit_concept_name = process_units(unit_concept_name)) 

# biomarker_df$visit_occurrence_concept_name[biomarker_df$visit_occurrence_concept_name %in% 
#                                            c("No matching concept", "NA")] <- NA

toc()

UACR2_df <- biomarker_df
rm(name, concept_ids_string)

In [ ]:
UACR2_df %>% count(unit_concept_name, unit_source_value, sort=T) 

In [ ]:
UACR2_df %>% count(unit_concept_name, sort=T) 

In [ ]:
UACR2_df %>% count( source_concept_name, source_concept_code, source_vocabulary, sort=T)

UACR2_df %>% 
    count(measurement_concept_id, standard_concept_name, source_concept_name, source_concept_code, source_vocabulary) %>% 
    arrange(standard_concept_name)

UACR2_df %>% count(operator_concept_name) %>% arrange(desc(n))

UACR2_df %>% filter(operator_concept_name == "<") %>% count(value_as_number, sort=T) %>% head()

UACR2_df %>% filter(operator_concept_name == ">") %>% count(value_as_number, sort=T) %>% head()

UACR2_df %>% count(range_high, sort=T) %>% filter(n>100)

UACR2_df %>% count(range_high, source_concept_name) %>% arrange(range_high) %>% filter(n>100)

UACR2_df %>% count(measurement_type_concept_name, sort=T) %>% filter(n>100)

UACR2_df %>% count(measurement_concept_id, standard_concept_name, sort=T)

In [ ]:
nrow(UACR2_df)
UACR2 <- UACR2_df %>%
        filter(unit_concept_name %in% c(
             NA,  "microg/mg of creatinine",  "mg/g", "mg/g of creatinine", "microg/mg", 
            "mg/mg of creatinine", "times",  "second",  "mcg/mg", "No matching concept",
            "NA", "no value",
            "microgram per milligram of creatinine", 
            "milligram per gram", 
            "milligram per gram of creatinine", 
            "microgram per milligram", 
            "milligram per milligram of creatinine", 
            "second"
        )) %>%
        select(-unit_source_value, -value_source_value) %>%
        mutate(unit_concept_name = gsub(" of creatinine", "", unit_concept_name)) %>%
        mutate(unit_concept_name = gsub("mg/g|mcg/mg|milligram per gram|microgram per milligram", 
                                        "microg/mg", unit_concept_name)) %>%
    filter(!is.na(value_as_number)) %>%
    filter(!(measurement_concept_id %in% 4154347)) %>%
    distinct()
nrow(UACR2)

In [ ]:
UACR2 <- UACR2 %>%
    filter(value_as_number >= 0.1 & value_as_number <= 20000) %>%
    dedup_records() %>% dedup_median() %>%
    mutate(value_as_number = signif(value_as_number, 3)) %>% 
    filter(is.na(mult_type_summary_flag))

nrow(UACR2)
fivenum(UACR2$value_as_number)
UACR2_summ <- UACR2 %>% group_by(person_id) %>% summarize(n=n())
fivenum(UACR2_summ$n)
nrow(UACR2_summ)

In [ ]:
write_to_bucket(UACR2, "bm_UACRdirect.csv")

In [ ]:
#also add the records from uring albumin query
poss_UACR_from_Albumin %>% count(measurement_concept_id, standard_concept_name, sort=T)

poss_UACR_from_Albumin %>% mutate(tp = "poss") %>%
    bind_rows(UACR2) %>%
    ggplot(aes(x=value_as_number, color = tp)) + geom_density() + 
    scale_x_log10()

In [ ]:
poss_UACR_from_Albumin %>% 
    group_by(visit_occurrence_concept_name) %>% filter(n() > 30) %>%
    filter(value_as_number < 20000) %>%
    ggplot(aes(x=value_as_number, color = visit_occurrence_concept_name)) + 
    geom_density(bw=.1) + 
    scale_x_log10()

In [ ]:
UACRcomb <- bind_rows(UACR2, poss_UACR_from_Albumin) %>%
    filter(value_as_number >= .1 & value_as_number <= 20000) %>%
    dedup_records() %>% dedup_median()

#### Combine calculated and directly pulled UACR

In [ ]:
UACR2 <- UACRcomb

UACR_combined <- UACR2 %>%
    dplyr::rename(UACR = value_as_number) %>%
    full_join(uacr_calculated)

In [ ]:
#Where the units disagree, use the smaller value.  Better to assume normal result than extremely abnormal value
UACR_combined_final <- UACR_combined %>% 
    filter(!(is.na(UACR) & !is.na(UACR_calculated) & UACR_calculated > 30)) %>% # don't abnormal calculated values
    mutate(UACR = pmin(UACR, UACR_calculated, na.rm=T),
          unit_concept_name = coalesce(unit_concept_name, 
                                       paste0(unit_concept_name_albumin, " / ", unit_concept_name_creatinine)),
          standard_concept_name = coalesce(standard_concept_name, 
                                           paste0(standard_concept_name_albumin, " / ", standard_concept_name_creatinine)),
          visit_occurrence_concept_name = coalesce(visit_occurrence_concept_name, 
                                           paste0(visit_occurrence_concept_name_albumin, " / ", visit_occurrence_concept_name_creatinine)),
          ) %>%
    filter(!is.na(UACR)) %>%
    distinct(person_id, measurement_date, UACR,
             urine_albumin, urine_creatinine,
             unit_concept_name,
             standard_concept_name, 
             measurement_concept_id, measurement_concept_id_albumin, measurement_concept_id_creatinine, 
             summarized_concept_ids_creatinine, mult_type_summary_flag_creatinine)

In [ ]:
write_to_bucket(UACR_combined_final, "bm_UACR.csv")

In [ ]:
rm(Albumin_urine, Albumin_urine_df, Creatinine_urine, Creatinine_urine_df, UACR, uacr_calculated, UACR_check,
   UACR_combined, UACR_combined_final, UACR_df)
gc()

### INR (International Normalized Ratio for prothrombin time)

In [ ]:
INR_df <- pull_lab(bm_concepts_key["INR"], name = "INR")                          
dim(INR_df)

In [ ]:
INR_df %>% count(unit_concept_name, sort=T)

INR_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

In [ ]:
INR_df %>% group_by(unit_concept_name) %>% filter(n() > 400) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(0.5, 5.5) +
    facet_wrap(~standard_concept_name)

There seems to be a bimodal distribution across units and concept ids. 

The main reason for using platelet-poor plasma for INR measurement is to standardize the test conditions and reduce variability. Platelets can release substances that affect coagulation, so removing them helps to ensure that the INR reflects only the clotting factors present in the plasma. This standardization allows for more consistent and accurate monitoring of patients receiving anticoagulant therapy, such as warfarin.

The bimodal distribution could be due to some missclassification between sample types (blood vs. PPP) but there is no way to clean this. 

In [ ]:
INR <- INR_df %>%
    filter(unit_concept_name %in% c(NA, "ratio", "second", "siemens", "times")) %>% 
    #seems like siemens is the brand name of the coagulation reagent? looks ok
    mutate(unit_concept_name = ifelse(unit_concept_name %in% c("siemens", "times"), 
                                      "second", unit_concept_name)) %>% #assume same 
    filter(value_as_number >= 0.5 & value_as_number <= 5.5) %>%
    dedup_records() %>% dedup_median()

In [ ]:
INR %>% group_by(standard_concept_name) %>% filter(n() > 1500) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() +
    facet_wrap(~standard_concept_name, scales="free_y") + 
    theme(legend.position="bottom") + 
    guides(color=guide_legend(ncol=1))

In [ ]:
nrow(INR)
fivenum(INR$value_as_number)
INR_summ <- INR %>% group_by(person_id) %>% summarize(n=n())
fivenum(INR_summ$n)
nrow(INR_summ)

In [ ]:
write_to_bucket(INR, "bm_INR.csv")

In [ ]:
rm(INR, INR_df)
gc()

### Prothrombin Time

In [ ]:
Prothrombin_time_df <- pull_lab(bm_concepts_key["Prothrombin_time"], name = "Prothrombin_time")                          
dim(Prothrombin_time_df)

In [ ]:
Prothrombin_time_df %>% count(unit_concept_name, sort=T)

Prothrombin_time_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(5,85)

Prothrombin_time_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

Prothrombin_time_df %>% group_by(standard_concept_name) %>% 
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(5,85)

In [ ]:
Prothrombin_time <- Prothrombin_time_df %>%
    filter(unit_concept_name %in% c("second", "seconds", NA, "siemens")) %>% 
    filter(value_as_number >= 5 & value_as_number <= 85) %>% 
    dedup_records() %>% dedup_median()

nrow(Prothrombin_time)
fivenum(Prothrombin_time$value_as_number)
Prothrombin_time_summ <- Prothrombin_time %>% group_by(person_id) %>% summarize(n=n())
fivenum(Prothrombin_time_summ$n)
nrow(Prothrombin_time_summ)

In [ ]:
write_to_bucket(Prothrombin_time, "bm_Prothrombin_time.csv")

In [ ]:
rm(Prothrombin_time, Prothrombin_time_df)
gc()

### Troponin

In [ ]:
Troponin_df <- pull_lab(bm_concepts_key["Troponin"], name = "Troponin")
dim(Troponin_df)

In [ ]:
Troponin_df %>% count(unit_concept_name, sort=T)

Troponin_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + scale_x_log10()

In [ ]:
# convert many units to ng/mL
Troponin <- Troponin_df %>% 
    filter(unit_concept_name %in% c( "ng/ml", "ng/l", NA, "picog/ml", 
                                    "microg/l", #??
                                    "ng/dl")) %>%
    mutate(value_as_number = ifelse( is.na(unit_concept_name), value_as_number, 
                                    ifelse( unit_concept_name %in% c("ng/dl"), 0.01*value_as_number, 
                                        ifelse(unit_concept_name %in% c("ng/l", "picog/ml"), 0.001*value_as_number, 
                                         value_as_number)))) %>%
    mutate(unit_concept_name =  ifelse( unit_concept_name %in% c("ng/dl", "ng/l", "picog/ml"), 
                                       "ng/mL", unit_concept_name))

In [ ]:
Troponin %>% count(measurement_concept_id, standard_concept_name, sort = T)

Troponin %>% group_by(standard_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density() + xlim(0, 0.4) +
    theme(legend.position ="bottom") + guides(color = guide_legend(ncol=1))

In [ ]:
Troponin <- Troponin %>%
    filter(!(measurement_concept_id %in% c(4005525, 4020703))) %>%
    filter(value_as_number >= 0 & value_as_number <= 0.7) %>% 
    dedup_records() %>% dedup_median()

nrow(Troponin)
fivenum(Troponin$value_as_number)
Troponin_summ <- Troponin %>% group_by(person_id) %>% summarize(n=n())
fivenum(Troponin_summ$n)
nrow(Troponin_summ)

In [ ]:
write_to_bucket(Troponin, "bm_Troponin.csv")

In [ ]:
rm(Troponin, Troponin_df)
gc()

### TotalCK

In [ ]:
TotalCK_df <- pull_lab(bm_concepts_key["TotalCK"], name = "TotalCK")                            
dim(TotalCK_df)

In [ ]:
TotalCK_df %>% count(unit_concept_name, sort=T)

TotalCK_df %>% group_by(unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density() + xlim(0,2000)

TotalCK_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

In [ ]:
TotalCK <- TotalCK_df %>%
    filter(unit_concept_name %in% c("unit/l",NA,"international unit/l","u/l","iu/l","nl" )) %>%
    filter(value_as_number >= 5 & value_as_number <= 2000) %>%
    dedup_records() %>% dedup_median()

nrow(TotalCK)
fivenum(TotalCK$value_as_number)
TotalCK_summ <- TotalCK %>% group_by(person_id) %>% summarize(n=n())
fivenum(TotalCK_summ$n)
nrow(TotalCK_summ)

In [ ]:
write_to_bucket(TotalCK, "bm_TotalCK.csv")

In [ ]:
rm(TotalCK, TotalCK_df)
gc()

### Uric Acid (Urate)

In [ ]:
bm_concepts_key["Uric_Acid"]
Uric_Acid_df <- pull_lab(bm_concepts_key["Uric_Acid"], name = "Uric_Acid")
dim(Uric_Acid_df)

In [ ]:
Uric_Acid_df %>% count(unit_concept_name, sort=T)
Uric_Acid_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

In [ ]:
Uric_Acid <- Uric_Acid_df %>%
    filter(unit_concept_name %in% c("mg/dl", NA, "mg/ml")) 

In [ ]:
Uric_Acid %>% group_by(standard_concept_name, unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name, linetype=unit_concept_name)) + 
    geom_density() + xlim(0, 20) 

In [ ]:
Uric_Acid <- Uric_Acid %>%
    filter(!(measurement_concept_id %in% c(4084176, 4199034))) %>%
    filter(value_as_number >= 0.5 & value_as_number <= 20) %>%
    dedup_records() %>% dedup_median()

In [ ]:
nrow(Uric_Acid_df)
nrow(Uric_Acid)
fivenum(Uric_Acid$value_as_number)
Uric_Acid_summ <- Uric_Acid %>% group_by(person_id) %>% summarize(n=n())
fivenum(Uric_Acid_summ$n)
nrow(Uric_Acid_summ)

In [ ]:
write_to_bucket(Uric_Acid, "bm_Uric_Acid.csv")

In [ ]:
rm(Uric_Acid_df, Uric_Acid)
gc()

### Bilirubin, conjugated/direct (blood/serum/plasma) (glucuronidated+albumin-bound)

In [ ]:
bm_concepts_key["Bilirubin_bsp_conjugated"] 

In [ ]:
Bilirubin_bsp_conjugated_df <- pull_lab(bm_concepts_key["Bilirubin_bsp_conjugated"], name = "Bilirubin_bsp_conjugated")
dim(Bilirubin_bsp_conjugated_df)

In [ ]:
Bilirubin_bsp_conjugated_df %>% count(unit_concept_name, sort=T)
Bilirubin_bsp_conjugated_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

In [ ]:
Bilirubin_bsp_conjugated <- Bilirubin_bsp_conjugated_df %>%
    filter(!(measurement_concept_id %in% c(3035521, 3044599))) %>% 
    filter(unit_concept_name %in% c("mg/dl", NA, "mg/ml")) 

In [ ]:
Bilirubin_bsp_conjugated %>% filter(value_as_number > 0) %>%
    mutate(id = factor(measurement_concept_id)) %>%
    group_by(standard_concept_name, unit_concept_name) %>% filter(n() > 150) %>%
    ggplot(aes(x=log10(value_as_number), color = unit_concept_name, linetype = id)) + 
    geom_density(bw = 0.1) + xlim(-1.5, log10(1)) +
    theme(legend.position = "bottom") +
    guides(color = guide_legend(ncol=1), linetype = guide_legend(ncol=1))

There are bimodal peaks, this may be due to different assays e.g. including albumin-bound vs. not.
See if we can see more details by visit type:

In [ ]:
Bilirubin_bsp_conjugated %>% #filter(value_as_number > 0) %>%
    mutate(id = factor(measurement_concept_id)) %>%
    group_by(standard_concept_name, unit_concept_name, visit_occurrence_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=log10(value_as_number + 0.01), color = unit_concept_name, linetype = id)) + 
    geom_density(bw = 0.1) + xlim(-1.5, log10(3)) +
    theme(legend.position = "bottom") +
    facet_wrap(~visit_occurrence_concept_name) + 
    guides(color = guide_legend(ncol=1), linetype = guide_legend(ncol=1))

The pattern persists across all visit types

In [ ]:
Bilirubin_bsp_conjugated %>% count(measurement_concept_id, unit_concept_name, value_as_number) %>%    
    group_by(measurement_concept_id, unit_concept_name) %>%
    arrange(measurement_concept_id, unit_concept_name, desc(n)) %>% 
    filter(row_number() %in% 1:5) 

In [ ]:
Bilirubin_bsp_conjugated <- Bilirubin_bsp_conjugated %>%
    filter(value_as_number >= 0 & value_as_number <= 20) %>%
    dedup_records() %>% dedup_median()

In [ ]:
nrow(Bilirubin_bsp_conjugated_df)
nrow(Bilirubin_bsp_conjugated)
fivenum(Bilirubin_bsp_conjugated$value_as_number)
Bilirubin_bsp_conjugated_summ <- Bilirubin_bsp_conjugated %>% group_by(person_id) %>% summarize(n=n())
fivenum(Bilirubin_bsp_conjugated_summ$n)
nrow(Bilirubin_bsp_conjugated_summ)

In [ ]:
write_to_bucket(Bilirubin_bsp_conjugated, "bm_Bilirubin_bsp_conjugated.csv")

In [ ]:
rm(Bilirubin_bsp_conjugated_df, Bilirubin_bsp_conjugated)
gc()

### Bilirubin, total (blood/serum/plasma)

In [ ]:
bm_concepts_key["Bilirubin_bsp_total"]

In [ ]:
Bilirubin_bsp_total_df <- pull_lab(bm_concepts_key["Bilirubin_bsp_total"], name = "Bilirubin_bsp_total")
dim(Bilirubin_bsp_total_df)

In [ ]:
Bilirubin_bsp_total_df %>% count(unit_concept_name, sort=T)
Bilirubin_bsp_total_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

In [ ]:
Bilirubin_bsp_total <- Bilirubin_bsp_total_df %>%
    filter(!(measurement_concept_id %in% c(
                3006140 # Bilirubin.total [Moles/volume] in Serum or Plasma	211
        ))) %>% 
    filter(unit_concept_name %in% c(NA, "mg/dl", "mg/ml")) 

In [ ]:
Bilirubin_bsp_total %>% 
    ggplot(aes(x=value_as_number, color = unit_concept_name)) + 
    geom_density(bw=.05) + xlim(0.1, 2)

Bilirubin_bsp_total %>% 
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    geom_density(bw=.05) + xlim(0.1, 2) +
    theme(legend.position = "bottom") +
    guides(color = guide_legend(ncol=1))

In [ ]:
Bilirubin_bsp_total <- Bilirubin_bsp_total %>%
    filter(value_as_number >= 0.1 & value_as_number <= 30) %>%
    dedup_records() %>% dedup_median()

In [ ]:
nrow(Bilirubin_bsp_total_df)
nrow(Bilirubin_bsp_total)
fivenum(Bilirubin_bsp_total$value_as_number)
Bilirubin_bsp_total_summ <- Bilirubin_bsp_total %>% group_by(person_id) %>% summarize(n=n())
fivenum(Bilirubin_bsp_total_summ$n)
nrow(Bilirubin_bsp_total_summ)

In [ ]:
write_to_bucket(Bilirubin_bsp_total, "bm_Bilirubin_bsp_total.csv")

In [ ]:
rm(Bilirubin_bsp_total_df, Bilirubin_bsp_total)
gc()

### Bilirubin, urine test strip

In [ ]:
Bilirubin_urine_df <- pull_lab(bm_concepts_key["Bilirubin_urine"], name = "Bilirubin_urine")
dim(Bilirubin_urine_df)

In [ ]:
Bilirubin_urine_df %>% count(unit_concept_name, sort=T)
Bilirubin_urine_df %>% count(measurement_concept_id, standard_concept_name, sort = T)

In [ ]:
Bilirubin_urine <- Bilirubin_urine_df %>%
    filter(measurement_concept_id %in% c(
            3018834, # Bilirubin.total [Presence] in Urine by Test strip
            3011258, # Bilirubin.total [Presence] in Urine
            3030477 # Bilirubin.total [Presence] in Urine by Automated test strip
       )) %>% 
    filter(unit_concept_name %in% c(NA,  "mg/dl", "ehrlich unit/dl", "mg/ml", "standardized quality unit" )) 

In [ ]:
Bilirubin_urine %>% group_by(standard_concept_name, unit_concept_name) %>% filter(n() > 100) %>%
    ggplot(aes(x=value_as_number, color = standard_concept_name)) + 
    facet_wrap(~unit_concept_name) +
    geom_density(bw=0.01) + xlim(0,6) +
    theme(legend.position = "bottom") +
    guides(color = guide_legend(ncol=1))

In [ ]:
Bilirubin_urine <- Bilirubin_urine %>%
    filter(value_as_number <= 10)

Only keep the stick/categorical results, discard continuous. 

In [ ]:
Bilirubin_urine <- Bilirubin_urine %>%
    filter(!is.na(unit_concept_name) | measurement_concept_id != 3011258) %>% 
    filter(value_as_number %in% c(0, 1, 2, 3, 4)) %>%
    distinct()

nrow(Bilirubin_urine)

In [ ]:
table(Bilirubin_urine$value_as_number)

Combine 3 and 4 together as 'high' category 

In [ ]:
Bilirubin_urine <- Bilirubin_urine %>%
    mutate(value_as_number = ifelse(value_as_number == 4, 3, value_as_number)) %>%
    distinct()

table(Bilirubin_urine$value_as_number)

In [ ]:
Bilirubin_urine <- Bilirubin_urine %>%
    dedup_records() %>% dedup_median()

In [ ]:
table(Bilirubin_urine$value_as_number)

In [ ]:
Bilirubin_urine <- Bilirubin_urine %>%
    mutate(value_as_number = ifelse(value_as_number %in% c(0.5, 1.5, 2.5), value_as_number + 0.5, value_as_number))

In [ ]:
nrow(Bilirubin_urine_df)
nrow(Bilirubin_urine)
fivenum(Bilirubin_urine$value_as_number)
Bilirubin_urine_summ <- Bilirubin_urine %>% group_by(person_id) %>% summarize(n=n())
fivenum(Bilirubin_urine_summ$n)
nrow(Bilirubin_urine_summ)

In [ ]:
write_to_bucket(Bilirubin_urine, "bm_Bilirubin_UrineStrip.csv")

In [ ]:
rm(Bilirubin_urine_df, Bilirubin_urine)
gc()

# Clean up

In [ ]:
toc()

In [ ]:
rm(list = ls())
gc()